# No-reference geometric quality assessment of raw 3D point clouds
**Complete, reproducible pipeline** for the article *"No-reference geometric quality assessment of raw 3D point clouds validated against independent reconstruction outcomes"* (B. Bhuvanavathi, T. Kamaleshwar, G. R. Kanagachidambaresan).

| Stage | Content | Hardware | Time (4 CPUs) |
|---|---|---|---|
| 1 | Controlled benchmark: 150 Thingi10K + 4 Stanford models × 46 variants, descriptors, frozen Poisson/ball-pivoting ground truth, closed-loop corrections | CPU | ≈ 14 h (resumable) |
| 2 | Benchmark analysis: fusion, cross-validation, specificity, attribution, sensitivity, runtime, MSND–curvature | CPU | ≈ 30 min |
| 3 | Real-data validation on DTU MVS 2014 (frozen weights), 22 standard scenes | CPU | ≈ 1 h |
| 3b | Extended DTU validation on all 80 scenes with all three MVS reconstructions (240 clouds) | CPU | ≈ 2–4 h |
| 4 | Correction policies, noise-estimator baselines, DTU robustness | CPU | ≈ 2–3 h |
| 5 | Learned baselines: PointNet (2048 / 8192 points) and hybrid PointNet + descriptors | **GPU** | ≈ 3 h |
| 6 | Exploratory unsupported surface fraction (USF) | CPU | ≈ 2.5 h |
| 7 | All figures 1–10 and S1 (600 dpi PNG + TIFF) | CPU | ≈ 5 min |

**Settings:** Internet **On**. Accelerator **None** for stages 1–4 (incl. 3b), 6, 7; **GPU** for stage 5.

**Kaggle sessions are limited to 12 h**, so run the stages over several sessions: set the `RUN` switches below, *Save Version → Save & Run All*, then attach that version's output as an **Input** for the next session. Every stage skips work whose results already exist (in `/kaggle/working` or any attached input).

All data are downloaded automatically (Thingi10K from Hugging Face, Stanford models, DTU from the DTU server). Seeds are fixed, so all clouds are regenerated exactly.

In [ ]:
# ---- stage switches (set True for the stages to run in this session) ----
SPLIT = None   # None = all objects; 0 / 1 = half of Stage 1 (two-account run)
RUN = dict(benchmark=True, analysis=True, dtu=True, dtu_extended=True, extras=True, pointnet=False, usf=True, figures=True)
N_THINGI, CLOSED_LOOP_N, N_MIXED, SEED = 150, 40, 15, 42
!pip install -q open3d trimesh rtree remotezip stream-unzip
import os, glob, zipfile, shutil, time, numpy as np, pandas as pd
os.environ['OMP_NUM_THREADS'] = '1'
W = '/kaggle/working'
D_ROWS, D_RES, D_DTU, D_DTU_RES, D_EXTRA, D_PN, D_USF, D_FIG, D_DTUX = [f'{W}/{d}' for d in ('rows','results','dtu_rows','dtu_results','extras','pointnet','usf','figures','dtu_extended')]
for d in (D_ROWS, D_RES, D_DTU, D_DTU_RES, D_EXTRA, D_PN, D_USF, D_FIG, D_DTUX): os.makedirs(d, exist_ok=True)
# resume: unpack attached zips and copy finished per-object/per-scan files from earlier sessions
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True): zipfile.ZipFile(z).extractall(f'{W}/unzipped')
def restore(sub, dst):
    for f in glob.glob(f'/kaggle/input/**/{sub}/*.csv', recursive=True) + glob.glob(f'{W}/unzipped/**/{sub}/*.csv', recursive=True):
        if not os.path.exists(os.path.join(dst, os.path.basename(f))): shutil.copy(f, dst)
for sub, dst in [('rows', D_ROWS), ('dtu_rows', D_DTU), ('dtu_extended_rows', D_DTUX+'/rows'), ('usf_rows', D_USF+'/usf_rows'), ('usf_dtu', D_USF+'/usf_dtu')]:
    os.makedirs(dst, exist_ok=True); restore(sub, dst)
def find(name):
    hits = sorted(glob.glob(f'{W}/**/{name}', recursive=True)) + sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None
print(os.cpu_count(), 'CPUs | finished benchmark objects:', len([f for f in os.listdir(D_ROWS) if '__loop' not in f]))

## Modules

In [ ]:
%%writefile prq_core.py
# =====================================================================
# prq_core.py  -  Print-Readiness Quality (PRQ) of raw point clouds
# Redesigned study: printer-aware NR metrics + independent
# reconstruction-based ground truth (frozen pipeline) + learned fusion.
# All geometry is normalised so the object's largest extent = PRINT_SIZE mm.
# =====================================================================
import numpy as np, time, math, warnings
from scipy.spatial import cKDTree
import open3d as o3d
import trimesh

warnings.filterwarnings("ignore")
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

CFG = dict(
    N_POINTS=20000,      # clean cloud size (Poisson-disk)
    PRINT_SIZE=100.0,    # mm, largest bbox extent after normalisation
    NOZZLE=0.4,          # mm  (printer spec -> metric reference scales)
    LAYER=0.2,           # mm
    SAMPLING_TARGET=1.0, # mm, target point spacing for reconstruction
    N_QUERY=5000,        # query points for per-point statistics
    TAU=1.0,             # mm, F-score threshold (primary)
    TAUS=(0.5, 1.0, 2.0),
    POISSON_DEPTH=8,
    N_EVAL=60000,        # GT / recon surface samples for evaluation
    RUN_BPA=True,        # second reconstruction pipeline (target robustness)
    K_NOISE=(12, 24),    # two scales for noise extrapolation
    K_BND=16, BND_THR=2 * 3.141592653589793 / 3,
    SEED=42,
)

# ---------------------------------------------------------------- utils
def normalize_mesh(V, F, size=None):
    size = size or CFG["PRINT_SIZE"]
    V = np.asarray(V, float)
    V = V - (V.max(0) + V.min(0)) / 2.0
    V = V * (size / (V.max(0) - V.min(0)).max())
    return V, np.asarray(F, np.int64)

def o3d_mesh(V, F):
    m = o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(V),
                                  o3d.utility.Vector3iVector(F))
    m.remove_duplicated_vertices(); m.remove_degenerate_triangles()
    m.remove_unreferenced_vertices()
    return m

def sample_clean(V, F, n=None, seed=0):
    n = n or CFG["N_POINTS"]
    m = o3d_mesh(V, F)
    try: o3d.utility.random.seed(seed)      # deterministic sampling (reproducibility)
    except Exception: pass
    pc = m.sample_points_poisson_disk(n, init_factor=4)
    return np.asarray(pc.points)

def sample_surface(V, F, n, seed=0):
    tm = trimesh.Trimesh(V, F, process=False)
    pts, _ = trimesh.sample.sample_surface(tm, n, seed=seed)
    return np.asarray(pts)

def _chunks(n, size=4000):
    for s in range(0, n, size):
        yield s, min(n, s + size)

def batched_pca(X):
    """X: (M,k,3) neighbourhoods -> centroid (M,3), eigvals asc (M,3), eigvecs (M,3,3)."""
    mu = X.mean(1)
    Xc = X - mu[:, None, :]
    C = np.einsum("mki,mkj->mij", Xc, Xc) / X.shape[1]
    w, v = np.linalg.eigh(C)
    return mu, w, v

def all_normals(P, tree, k=16):
    N = np.empty_like(P)
    for s, e in _chunks(len(P)):
        _, ix = tree.query(P[s:e], k=k, workers=-1)
        _, _, v = batched_pca(P[ix])
        N[s:e] = v[:, :, 0]
    return N

# ---------------------------------------------------------------- corruptions
# Each corruption takes severity s in [0,1]; s=0 means "lowest non-zero level".
RANGES = dict(
    noise=(0.03, 1.0),        # Gaussian sigma, mm  (layer = 0.2 mm)
    sparsity=(0.7, 0.04),     # keep fraction
    nonuniform=(0.3, 0.97),   # density contrast of smooth random field
    holes=(0.01, 0.20),       # removed surface-area fraction (disk patches)
    outliers=(0.002, 0.10),   # outlier fraction (uniform + floating clusters)
    ghosting=(0.2, 4.0),      # mm offset of duplicated mis-registered patch (HELD-OUT type)
)
TYPES = ["noise", "sparsity", "nonuniform", "holes", "outliers"]  # used in mixed
LEVELS = [0.0, 0.25, 0.5, 0.75, 1.0]

def _geo(lo, hi, s):  return lo * (hi / lo) ** s
def _lin(lo, hi, s):  return lo + (hi - lo) * s

def c_noise(P, s, rng):
    sig = _geo(*RANGES["noise"], s)
    return P + rng.normal(0, sig, P.shape), dict(noise_sigma=sig)

def c_sparsity(P, s, rng):
    keep = _geo(*RANGES["sparsity"], s)
    idx = rng.choice(len(P), max(200, int(keep * len(P))), replace=False)
    return P[idx], dict(keep=keep)

def c_nonuniform(P, s, rng):
    a = _lin(*RANGES["nonuniform"], s)
    size = (P.max(0) - P.min(0)).max()
    C = P[rng.choice(len(P), 4, replace=False)]
    r = 0.25 * size
    f = np.exp(-((P[:, None, :] - C[None]) ** 2).sum(-1) / (2 * r * r)).sum(1)
    f = (f - f.min()) / (f.max() - f.min() + 1e-12)
    keep = rng.random(len(P)) < (1 - a * f)
    return P[keep], dict(contrast=a)

def c_holes(P, s, rng):
    frac = _geo(*RANGES["holes"], s)
    nh = int(rng.integers(1, 4))
    m = max(10, int(frac * len(P) / nh))
    tree = cKDTree(P)
    rm = np.zeros(len(P), bool)
    for _ in range(nh):
        _, ix = tree.query(P[rng.integers(len(P))], k=m)
        rm[ix] = True
    return P[~rm], dict(hole_frac=frac, n_holes=nh)

def c_outliers(P, s, rng):
    frac = _geo(*RANGES["outliers"], s)
    n = int(frac * len(P)); nu = n // 2; nc = n - nu
    lo, hi = P.min(0), P.max(0); ext = hi - lo
    U = rng.uniform(lo - 0.1 * ext, hi + 0.1 * ext, (nu, 3))
    centers = P[rng.choice(len(P), 5)] + rng.normal(0, 1, (5, 3)) * 5.0   # floaters ~5 mm off surface
    Cl = centers[rng.integers(0, 5, nc)] + rng.normal(0, 1.5, (nc, 3))
    return np.vstack([P, U, Cl]), dict(out_frac=frac)

def c_ghosting(P, s, rng):
    d = _geo(*RANGES["ghosting"], s)
    tree = cKDTree(P)
    _, ix = tree.query(P[rng.integers(len(P))], k=int(0.3 * len(P)))
    patch = P[ix]
    _, _, v = batched_pca(patch[None])
    n = v[0, :, 0]
    return np.vstack([P, patch + d * n]), dict(ghost_mm=d)

CORR = dict(noise=c_noise, sparsity=c_sparsity, nonuniform=c_nonuniform,
            holes=c_holes, outliers=c_outliers, ghosting=c_ghosting)

def make_variants(P0, rng, n_mixed=15):
    """Clean + single-factor (6 types x 5 levels) + mixed continuous variants."""
    out = [("clean", "clean", 0.0, P0.copy(), {})]
    for t in CORR:
        for s in LEVELS:
            P, meta = CORR[t](P0, s, rng)
            out.append((f"{t}_s{s:.2f}", t, s, P, meta))
    for j in range(n_mixed):
        act = [t for t in TYPES if rng.random() < 0.5]
        if len(act) < 2: act = list(rng.choice(TYPES, 2, replace=False))
        P, meta = P0.copy(), {}
        sev = {}
        for t in ["holes", "sparsity", "nonuniform", "noise", "outliers"]:  # fixed physical order
            if t in act:
                sv = float(rng.random()); sev[t] = sv
                P, m = CORR[t](P, sv, rng); meta.update(m)
        meta.update({f"sev_{k}": v for k, v in sev.items()})
        out.append((f"mixed_{j:02d}", "mixed", float(np.mean(list(sev.values()))), P, meta))
    return out

# ---------------------------------------------------------------- NR metrics (proposed)
def _score(raw, ref):
    """Smooth, never-saturating map: 1 at raw=0, 0.5 at raw=ref."""
    return float(2.0 ** (-max(raw, 0.0) / ref))

REFS = dict(  # printer-aware / physically interpretable reference scales
    spacing_excess=lambda: CFG["SAMPLING_TARGET"],  # mm beyond target spacing
    density_logstd=lambda: 0.5,                     # robust std of log surface density
    noise_mm=lambda: CFG["LAYER"] / 2,              # half a layer height
    boundary_frac=lambda: 0.02,
    outlier_frac=lambda: 0.01,
    normal_dev=lambda: 0.15,                        # rad
    msnd=lambda: 0.05,
)

def proposed_metrics(P, rng=None, return_pointwise=False):
    rng = rng or np.random.default_rng(0)
    t0 = time.perf_counter()
    n = len(P)
    tree = cKDTree(P)
    q = rng.choice(n, min(n, CFG["N_QUERY"]), replace=False)
    Q = P[q]
    K = max(41, CFG['K_NOISE'][1] + 1)
    d, ix = tree.query(Q, k=K, workers=-1)          # col 0 = self
    d, ix = d[:, 1:], ix[:, 1:]
    out = {}

    # M1 sampling sufficiency: median spacing (mean 6-NN), printer-relative
    sp = d[:, :6].mean(1)
    h = float(np.median(sp))
    out["raw_spacing_mm"] = h
    out["s_spacing"] = _score(h - CFG["SAMPLING_TARGET"], REFS["spacing_excess"]())

    # M2 density uniformity: robust std of log surface density at 2 scales (unclipped)
    def logstd(k):
        rho = np.log(k / (np.pi * d[:, k - 1] ** 2 + 1e-12))
        q75, q25 = np.percentile(rho, [75, 25]); return (q75 - q25) / 1.349
    ls8, ls32 = logstd(8), logstd(32)
    out["raw_dens_fine"], out["raw_dens_coarse"] = ls8, ls32
    out["raw_density"] = 0.5 * (ls8 + ls32)
    out["s_density"] = _score(out["raw_density"], REFS["density_logstd"]())

    # M3 noise: SCALE-EXTRAPOLATED quadric residual (mm).
    # Quadric residual at radius r ~ sigma_n^2 + c*r^6 (unresolved shape); two scales
    # (k=12, k=24) let us cancel the shape term -> noise decoupled from sampling density.
    def quad_sigma(k):
        kk = np.concatenate([q[:, None], ix[:, :k - 1]], 1)
        u, ww, hh = _local_uvh(P[kk])
        A = np.stack([u**2, u*ww, ww**2, u, ww, np.ones_like(u)], -1)
        c = np.linalg.solve(np.einsum("mki,mkj->mij", A, A) + 1e-9*np.eye(6),
                            np.einsum("mki,mk->mi", A, hh)[..., None])[..., 0]
        r = hh - np.einsum("mki,mi->mk", A, c)
        return float(np.median(np.sqrt((r**2).sum(1) / (k - 6)))), float(np.median(d[:, k - 2]))
    ka, kb = CFG['K_NOISE']
    s12, r12 = quad_sigma(ka); s24, r24 = quad_sigma(kb)
    ratio = (r24 / max(r12, 1e-12)) ** 6
    var_n = (ratio * s12**2 - s24**2) / max(ratio - 1, 1e-9)
    out["raw_noise_q12_mm"], out["raw_noise_q24_mm"] = s12, s24
    out["raw_noise_mm"] = float(np.sqrt(max(var_n, 0.0)))
    out["s_noise"] = _score(out["raw_noise_mm"], REFS["noise_mm"]())

    # M4 holes: angle-gap boundary detector on ALL points, keep only boundary points that
    # form chains (>=2 boundary neighbours) -> hole rims, not isolated sampling gaps.
    bnd_all = _boundary_points(P, tree, CFG['K_BND'], CFG['BND_THR'])
    _, ixa = tree.query(P, k=9, workers=-1)
    chain = bnd_all & (bnd_all[ixa[:, 1:]].sum(1) >= 2)
    out["raw_boundary_frac"] = float(chain.mean())
    out["s_holes"] = _score(out["raw_boundary_frac"], REFS["boundary_frac"]())
    bnd = chain[q]

    # M5 outliers: ratio criterion on mean 8-NN distance over ALL points
    dall, _ = tree.query(P, k=9, workers=-1)
    m8 = dall[:, 1:].mean(1)
    of = float((m8 > 3.0 * np.median(m8)).mean())
    out["raw_outlier_frac"] = of
    out["s_outliers"] = _score(of, REFS["outlier_frac"]())

    # M6 normal consistency (k=16 PCA normals, |cos|)
    Nall = all_normals(P, tree, 16)
    nq = Nall[q]; nn = Nall[ix[:, :8]]
    cos = np.abs(np.einsum("mi,mki->mk", nq, nn)).clip(0, 1)
    out["raw_normal_dev"] = float(np.arccos(cos).mean())
    out["s_normals"] = _score(out["raw_normal_dev"], REFS["normal_dev"]())

    # M7 MSND multi-scale normal deviation (renamed from 'curvature')
    k10 = np.concatenate([q[:, None], ix[:, :9]], 1)
    k40 = np.concatenate([q[:, None], ix[:, :39]], 1)
    _, _, v10 = batched_pca(P[k10]); _, _, v40 = batched_pca(P[k40])
    msnd_pt = 1 - np.abs((v10[:, :, 0] * v40[:, :, 0]).sum(1))
    out["raw_msnd"] = float(msnd_pt.mean())
    out["s_msnd"] = _score(out["raw_msnd"], REFS["msnd"]())

    # hole extent (auxiliary, mm): largest connected cluster of boundary queries
    out["hole_extent_mm"] = _largest_cluster_extent(Q[bnd], 4 * h)
    out["n_points"] = n
    out["t_metrics_s"] = time.perf_counter() - t0
    if return_pointwise:
        return out, dict(q=q, msnd=msnd_pt, boundary=bnd)
    return out

def _local_uvh(X):
    mu, w, v = batched_pca(X); Xc = X - mu[:, None, :]
    return (np.einsum("mki,mi->mk", Xc, v[:, :, 1]), np.einsum("mki,mi->mk", Xc, v[:, :, 2]),
            np.einsum("mki,mi->mk", Xc, v[:, :, 0]))

def _boundary_points(P, tree, k=16, thr=2 * np.pi / 3):
    B = np.zeros(len(P), bool)
    for s, e in _chunks(len(P)):
        _, ix = tree.query(P[s:e], k=k, workers=-1)
        u, w, _ = _local_uvh(P[ix])
        ang = np.sort(np.arctan2(w[:, 1:] - w[:, :1], u[:, 1:] - u[:, :1]), 1)
        gaps = np.diff(np.concatenate([ang, ang[:, :1] + 2 * np.pi], 1), axis=1)
        B[s:e] = gaps.max(1) > thr
    return B

def _largest_cluster_extent(B, eps):
    if len(B) < 3: return 0.0
    t = cKDTree(B); pairs = t.query_pairs(eps, output_type="ndarray")
    par = np.arange(len(B))
    def f(i):
        while par[i] != i: par[i] = par[par[i]]; i = par[i]
        return i
    for a, b in pairs:
        ra, rb = f(a), f(b)
        if ra != rb: par[ra] = rb
    roots = np.array([f(i) for i in range(len(B))])
    best = 0.0
    for r_ in np.unique(roots):
        S = B[roots == r_]
        if len(S) > 2: best = max(best, float(np.linalg.norm(S.max(0) - S.min(0))))
    return best

PROPOSED = ["s_spacing", "s_density", "s_noise", "s_holes", "s_outliers", "s_normals", "s_msnd"]

# ---------------------------------------------------------------- previous CQI (manuscript v1) + conventional descriptors
def previous_metrics(P):
    n = len(P); tree = cKDTree(P)
    d, ix = tree.query(P, k=41, workers=-1); d, ix = d[:, 1:], ix[:, 1:]
    diag = float(np.linalg.norm(P.max(0) - P.min(0)))
    d5, d20 = d[:, :5].mean(1), d[:, :20].mean(1)
    cov = lambda x: x.std() / (x.mean() + 1e-12)
    o = {}
    o["old_density"] = float(np.clip(0.6 * (1 - cov(d5)) + 0.4 * (1 - cov(d20)), 0, 1))
    sub = np.random.default_rng(0).choice(n, min(n, CFG["N_QUERY"]), replace=False)
    X = P[np.concatenate([sub[:, None], ix[sub, :19]], 1)]
    mu, w, v = batched_pca(X)
    res = np.abs(np.einsum("mki,mi->mk", X - mu[:, None], v[:, :, 0])).mean()
    o["old_noise"] = float(max(0, 1 - res / (2.0 * d20.mean())))
    mx = d[:, 9]; rh = (mx > 3 * np.median(mx)).mean()
    o["old_holes"] = float(max(0, 1 - 3 * rh))
    N = all_normals(P, tree, 20)
    th = np.arccos(np.abs(np.einsum("mi,mki->mk", N[sub], N[ix[sub, :20]])).clip(0, 1)).mean()
    o["old_normals"] = float(max(0, 1 - 2 * th))
    ro = (d20 > d20.mean() + 2 * d20.std()).mean()
    o["old_outliers"] = float(max(0, 1 - 5 * ro))
    _, _, v10 = batched_pca(P[np.concatenate([sub[:, None], ix[sub, :9]], 1)])
    _, _, v40 = batched_pca(P[np.concatenate([sub[:, None], ix[sub, :39]], 1)])
    kk = (1 - np.abs((v10[:, :, 0] * v40[:, :, 0]).sum(1))).mean()
    o["old_curv"] = float(max(0, 1 - 2 * kk))
    o["CQI_v1"] = float(np.mean([o[k] for k in ["old_density", "old_noise", "old_holes",
                                                 "old_normals", "old_outliers", "old_curv"]]))
    # conventional single-scale descriptors (for descriptor ablation, Reviewer 3 #7)
    o["conv_density_cov5"] = float(cov(d5))
    o["conv_noise_bbox"] = float(res / diag)
    o["conv_curv_eigratio"] = float((w[:, 0] / w.sum(1)).mean())
    return o

# ---------------------------------------------------------------- full-reference baselines (privileged)
def fr_baselines(P, G, treeG=None):
    treeG = treeG or cKDTree(G)
    a, _ = treeG.query(P, workers=-1); b, _ = cKDTree(P).query(G, workers=-1)
    return dict(FR_chamfer=float(a.mean() + b.mean()),
                FR_hausdorff=float(max(a.max(), b.max())),
                FR_hausdorff95=float(max(np.percentile(a, 95), np.percentile(b, 95))))

# ---------------------------------------------------------------- independent ground truth (frozen pipeline)
def _pcd_with_normals(P):
    pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P))
    pc.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(30))
    try: pc.orient_normals_consistent_tangent_plane(10)
    except Exception: pass
    return pc

def reconstruct_poisson(P):
    pc = _pcd_with_normals(P)
    try:
        m, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pc, depth=CFG["POISSON_DEPTH"], scale=1.1, linear_fit=False, n_threads=1)
    except TypeError:
        m, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pc, depth=CFG["POISSON_DEPTH"], scale=1.1, linear_fit=False)
    return np.asarray(m.vertices), np.asarray(m.triangles)

def reconstruct_bpa(P):
    pc = _pcd_with_normals(P)
    h = float(np.median(cKDTree(P).query(P, k=2)[0][:, 1]))
    m = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pc, o3d.utility.DoubleVector([1.5 * h, 3 * h, 6 * h]))
    return np.asarray(m.vertices), np.asarray(m.triangles)

def evaluate_mesh(V, F, G, treeG, prefix, gt_euler=None, full_topology=True):
    o = {}
    if len(F) == 0:
        for t in CFG["TAUS"]: o[f"{prefix}F@{t}"] = 0.0
        o[f"{prefix}chamfer_mm"] = np.nan
        if full_topology:
            o.update({f"{prefix}watertight": 0, f"{prefix}winding_ok": 0,
                      f"{prefix}n_components": 0, f"{prefix}print_ready": 0})
        return o
    R = sample_surface(V, F, CFG["N_EVAL"], seed=1)
    a, _ = treeG.query(R, workers=-1); b, _ = cKDTree(R).query(G, workers=-1)
    for t in CFG["TAUS"]:
        p, r = (a < t).mean(), (b < t).mean()
        o[f"{prefix}F@{t}"] = float(2 * p * r / (p + r + 1e-12))
    o[f"{prefix}precision@{CFG['TAU']}"] = float((a < CFG["TAU"]).mean())
    o[f"{prefix}recall@{CFG['TAU']}"] = float((b < CFG["TAU"]).mean())
    o[f"{prefix}chamfer_mm"] = float(a.mean() + b.mean())
    if full_topology:
        tm = trimesh.Trimesh(V, F, process=True)
        o[f"{prefix}watertight"] = int(tm.is_watertight)
        o[f"{prefix}winding_ok"] = int(tm.is_winding_consistent)
        comps = tm.split(only_watertight=False)
        A = tm.area + 1e-12
        o[f"{prefix}n_components"] = int(sum(c.area > 0.01 * A for c in comps))
        o[f"{prefix}euler"] = int(tm.euler_number)
        if gt_euler is not None: o[f"{prefix}topology_match"] = int(tm.euler_number == gt_euler)
        o[f"{prefix}print_ready"] = int(o[f"{prefix}watertight"] and o[f"{prefix}winding_ok"]
                                        and o[f"{prefix}n_components"] == 1
                                        and o[f"{prefix}F@{CFG['TAU']}"] >= 0.90)
    return o

def downstream_target(P, G, treeG, gt_euler=None):
    o = {}
    t0 = time.perf_counter()
    try: V, F = reconstruct_poisson(P)
    except Exception: V, F = np.zeros((0, 3)), np.zeros((0, 3), int)
    o["t_poisson_s"] = time.perf_counter() - t0
    o.update(evaluate_mesh(V, F, G, treeG, "PSR_", gt_euler))
    if CFG["RUN_BPA"]:
        t0 = time.perf_counter()
        try: V, F = reconstruct_bpa(P)
        except Exception: V, F = np.zeros((0, 3)), np.zeros((0, 3), int)
        o["t_bpa_s"] = time.perf_counter() - t0
        o.update(evaluate_mesh(V, F, G, treeG, "BPA_", full_topology=False))
    return o

# ---------------------------------------------------------------- corrective actions (for closed loop)
def fix_denoise(P, k=24):
    tree = cKDTree(P); Pn = P.copy()
    for s, e in _chunks(len(P)):
        _, ix = tree.query(P[s:e], k=k, workers=-1)          # includes self at col 0
        X = P[ix]; mu, w, v = batched_pca(X); Xc = X - mu[:, None]
        u = np.einsum("mki,mi->mk", Xc, v[:, :, 1]); ww = np.einsum("mki,mi->mk", Xc, v[:, :, 2])
        hh = np.einsum("mki,mi->mk", Xc, v[:, :, 0])
        A = np.stack([u**2, u*ww, ww**2, u, ww, np.ones_like(u)], -1)
        c = np.linalg.solve(np.einsum("mki,mkj->mij", A, A) + 1e-9*np.eye(6),
                            np.einsum("mki,mk->mi", A, hh)[..., None])[..., 0]
        hfit = (A[:, 0] * c).sum(1)
        Pn[s:e] = P[s:e] + (hfit - hh[:, 0])[:, None] * v[:, :, 0]
    return Pn

def fix_outliers(P):
    pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P))
    pc, _ = pc.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    return np.asarray(pc.points)

def fix_uniformize(P):
    h = float(np.median(cKDTree(P).query(P, k=7)[0][:, 1:].mean(1)))
    # conservative: only thins over-dense clusters (voxel < median spacing), never coarsens
    pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P)).voxel_down_sample(0.8 * h)
    return np.asarray(pc.points)

FIXES = dict(outliers=fix_outliers, denoise=fix_denoise, uniformize=fix_uniformize)
FIX_ORDER = ["outliers", "denoise", "uniformize"]
# metric -> recommended action ('recapture' = no point-level fix possible)
ACTION_OF = dict(raw_outlier_frac="outliers", raw_noise_mm="denoise", raw_normal_dev="denoise",
                 raw_msnd="denoise", raw_density="uniformize",
                 raw_spacing_mm="recapture", raw_boundary_frac="recapture")
RAW = list(ACTION_OF)

def global_shape_descriptors(P):
    """Category-free descriptors used to condition fusion weights (no labels needed)."""
    X = P - P.mean(0); w = np.sort(np.linalg.eigvalsh(X.T @ X / len(P)))[::-1] + 1e-12
    return dict(g_logn=float(np.log10(len(P))), g_linearity=float((w[0]-w[1])/w[0]),
                g_planarity=float((w[1]-w[2])/w[0]), g_sphericity=float(w[2]/w[0]))


In [ ]:
%%writefile prq_run.py
# =====================================================================
# prq_run.py - data loading + per-object experiment (resumable, parallel)
# =====================================================================
import os, zlib, socket, itertools, urllib.request, tarfile, gzip, shutil, numpy as np, pandas as pd, trimesh
from scipy.spatial import cKDTree
import open3d as o3d
from prq_core import *

# ---------------------------------------------------------------- datasets
def load_stanford(cache="/kaggle/working/stanford", include_lucy=False):
    """Real structured-light scans. Bunny/Armadillo via Open3D; Dragon/Buddha/Lucy from Stanford repo."""
    os.makedirs(cache, exist_ok=True); objs = {}; socket.setdefaulttimeout(60)
    for name, ds in [("Bunny", "BunnyMesh"), ("Armadillo", "ArmadilloMesh")]:
        try:
            m = o3d.io.read_triangle_mesh(getattr(o3d.data, ds)().path)
            objs[name] = (np.asarray(m.vertices), np.asarray(m.triangles))
        except Exception as e: print("skip", name, e)
    urls = {"Dragon": ("http://graphics.stanford.edu/pub/3Dscanrep/dragon/dragon_recon.tar.gz", "dragon_vrip.ply"),
            "Buddha": ("http://graphics.stanford.edu/pub/3Dscanrep/happy/happy_recon.tar.gz", "happy_vrip.ply")}
    if include_lucy:
        urls["Lucy"] = ("http://graphics.stanford.edu/data/3Dscanrep/lucy.tar.gz", "lucy.ply")
    for name, (url, fname) in urls.items():
        try:
            tgz = os.path.join(cache, os.path.basename(url))
            if not os.path.exists(tgz): urllib.request.urlretrieve(url, tgz)
            with tarfile.open(tgz) as t:
                mem = [m for m in t.getmembers() if m.name.endswith(fname)][0]
                t.extract(mem, cache); path = os.path.join(cache, mem.name)
            m = o3d.io.read_triangle_mesh(path)
            if len(m.triangles) > 400000: m = m.simplify_quadric_decimation(400000)
            objs[name] = (np.asarray(m.vertices), np.asarray(m.triangles))
        except Exception as e: print("skip", name, e)
    return objs

# Thingi10K is fetched DIRECTLY from Hugging Face (metadata CSV + npz archive).
# The `thingi10k` pip package is NOT used: its dependency `lagrange` crashes on
# Kaggle's current image (TBB symbol conflict). Same data, same filters.
HF = "https://huggingface.co/datasets/Thingi10K/Thingi10K/resolve/v1.5.0"
CORRUPT = {49911, 74463, 286163, 77942}

def _fetch(url, dst, retries=3):
    if os.path.exists(dst) and os.path.getsize(dst) > 0: return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for k in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dst + ".part", "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 22)
            os.replace(dst + ".part", dst); return dst
        except Exception as e:
            print(f"download retry {k+1}/{retries}: {url} -> {e}")
    raise RuntimeError(f"could not download {url}")

def thingi_ids(n=150, seed=42, facets=(2000, 200000), cache="/tmp/thingi10k"):
    """Thingi10K: 10k real 3D-printing models. Filter to clean, printable, single-part solids."""
    geo = pd.read_csv(_fetch(f"{HF}/metadata/geometry_data.csv", f"{cache}/geometry_data.csv"))
    print("metadata columns:", list(geo.columns)[:12], "...")
    def col(name, want):              # tolerate a missing column instead of crashing
        if name not in geo: print("  (filter skipped, column missing:", name, ")"); return pd.Series(True, index=geo.index)
        return pd.to_numeric(geo[name], errors="coerce") == want
    nf = pd.to_numeric(geo.get("num_faces", geo.get("num_facets")), errors="coerce")
    m = (col("solid", 1) & col("vertex_manifold", 1) & col("edge_manifold", 1)
         & col("num_connected_components", 1) & col("num_self_intersections", 0)
         & col("num_boundary_edges", 0) & nf.between(*facets) & ~geo.file_id.isin(CORRUPT))
    cand = geo.loc[m, "file_id"].astype(int).sort_values().values
    print(f"Thingi10K: {len(geo)} models, {len(cand)} pass the printable-solid filter")
    pick = sorted(np.random.default_rng(seed).choice(cand, min(n, len(cand)), replace=False).tolist())
    # licence / thing id (for the data-availability statement)
    meta = {}
    try:
        summ = pd.read_csv(_fetch(f"{HF}/metadata/input_summary.csv", f"{cache}/input_summary.csv"))
        ctx = pd.read_csv(_fetch(f"{HF}/metadata/contextual_data.csv", f"{cache}/contextual_data.csv"))
        j = summ.merge(ctx, on="Thing ID", how="left").set_index("ID")
        meta = {int(i): (int(j.loc[i, "Thing ID"]), str(j.loc[i, "License"])) for i in pick if i in j.index}
    except Exception as e: print("metadata (licence) unavailable:", e)
    # mesh archive: download once, extract only the selected files
    out_dir = f"{cache}/npz"; os.makedirs(out_dir, exist_ok=True)
    need = {f"{i}.npz" for i in pick if not os.path.exists(f"{out_dir}/{i}.npz")}
    if need:
        tgz = _fetch(f"{HF}/Thingi10K_npz.tar.gz", f"{cache}/Thingi10K_npz.tar.gz")
        with tarfile.open(tgz, "r:gz") as t:
            for mem in t:
                b = os.path.basename(mem.name)
                if mem.isfile() and b in need:
                    with t.extractfile(mem) as src, open(f"{out_dir}/{b}", "wb") as dst: shutil.copyfileobj(src, dst)
        os.remove(tgz)                                   # free disk
    ents = []
    for i in pick:
        p = f"{out_dir}/{i}.npz"
        if os.path.exists(p):
            tid, lic = meta.get(i, (None, "unknown"))
            ents.append(dict(file_id=i, thing_id=tid, license=lic, file_path=p))
    print(f"Thingi10K: {len(ents)} meshes ready")
    return ents

def load_thingi(entry):
    with np.load(entry["file_path"]) as d:
        return np.asarray(d["vertices"], float), np.asarray(d["facets"], np.int64)

# ---------------------------------------------------------------- per-object experiment
CLOSED_LOOP_SETS = [c for r in range(1, 4) for c in itertools.combinations(FIX_ORDER, r)]

def _closed_loop_candidates(variants):
    keep = []
    for v in variants:
        name, typ, s = v[0], v[1], v[2]
        if typ in ("clean", "mixed") or (typ in ("noise", "outliers", "nonuniform") and s in (0.5, 1.0)):
            keep.append(v)
    return keep

def process_object(obj_id, V, F, source, out_dir, n_mixed=15, closed_loop=False, seed=42):
    os.makedirs(out_dir, exist_ok=True)
    f_main = os.path.join(out_dir, f"{obj_id}.csv")
    if os.path.exists(f_main): return f_main
    try:
        V, F = normalize_mesh(V, F)
        P0 = sample_clean(V, F)
        G = sample_surface(V, F, CFG["N_EVAL"], seed=0); treeG = cKDTree(G)
        tm = trimesh.Trimesh(V, F, process=True)
        gt_euler = int(tm.euler_number) if tm.is_watertight else None
        rng = np.random.default_rng(zlib.crc32(f"{obj_id}|{seed}".encode()))
        variants = make_variants(P0, rng, n_mixed)
        rows, loop_rows = [], []
        for name, typ, sev, P, meta in variants:
            r = dict(object=obj_id, source=source, variant=name, type=typ, severity=sev)
            r.update(meta); r.update(global_shape_descriptors(P))
            r.update(proposed_metrics(P)); r.update(previous_metrics(P))
            r.update(fr_baselines(P, G, treeG)); r.update(downstream_target(P, G, treeG, gt_euler))
            rows.append(r)
        if closed_loop:
            for name, typ, sev, P, meta in _closed_loop_candidates(variants):
                for combo in CLOSED_LOOP_SETS:
                    Q = P
                    for a in FIX_ORDER:
                        if a in combo and len(Q) > 300: Q = FIXES[a](Q)
                    lr = dict(object=obj_id, variant=name, actions="+".join(combo), n_after=len(Q))
                    pm = proposed_metrics(Q); lr.update({k: pm[k] for k in PROPOSED + RAW})
                    tgt = evaluate_mesh(*reconstruct_poisson(Q), G, treeG, "PSR_", gt_euler)
                    lr.update(tgt)
                    if CFG["RUN_BPA"]:   # pipeline without built-in smoothing
                        try: lr.update(evaluate_mesh(*reconstruct_bpa(Q), G, treeG, "BPA_", full_topology=False))
                        except Exception: pass
                    loop_rows.append(lr)
            pd.DataFrame(loop_rows).to_csv(os.path.join(out_dir, f"{obj_id}__loop.csv"), index=False)
        pd.DataFrame(rows).to_csv(f_main, index=False)
    except Exception as e:
        with open(os.path.join(out_dir, f"{obj_id}.err"), "w") as fh: fh.write(repr(e))
    return f_main


In [ ]:
%%writefile prq_analysis.py
# =====================================================================
# prq_analysis.py - benchmark analysis: fusion, cross-validation, specificity, attribution, closed loop
# Ground truth = frozen-pipeline reconstruction outcome (independent of CQI).
# =====================================================================
import os, glob, json, numpy as np, pandas as pd
from scipy import stats, optimize
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, cohen_kappa_score, accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingRegressor
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from prq_core import PROPOSED, RAW, ACTION_OF, FIX_ORDER

plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 600, "font.size": 9,
                     "font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
OLD = ["old_density", "old_noise", "old_holes", "old_normals", "old_outliers", "old_curv"]
CONV = ["conv_density_cov5", "conv_noise_bbox", "conv_curv_eigratio"]
GDESC = ["g_logn", "g_linearity", "g_planarity", "g_sphericity"]
DEFECT_OF = dict(raw_noise_mm="noise", raw_spacing_mm="sparsity", raw_density="nonuniform",
                 raw_boundary_frac="holes", raw_outlier_frac="outliers")
SINGLE = ["noise", "sparsity", "nonuniform", "holes", "outliers", "ghosting"]

def load(rows_dir):
    df = pd.concat([pd.read_csv(f) for f in glob.glob(f"{rows_dir}/*.csv") if "__loop" not in f], ignore_index=True)
    lp = [pd.read_csv(f) for f in glob.glob(f"{rows_dir}/*__loop.csv")]
    loop = pd.concat(lp, ignore_index=True) if lp else pd.DataFrame()
    df["RQ"] = df[["PSR_F@0.5", "PSR_F@1.0", "PSR_F@2.0"]].mean(1)       # primary continuous target
    if "BPA_F@1.0" in df: df["RQ_BPA"] = df[["BPA_F@0.5", "BPA_F@1.0", "BPA_F@2.0"]].mean(1)
    if len(loop):
        loop["RQ"] = loop[["PSR_F@0.5", "PSR_F@1.0", "PSR_F@2.0"]].mean(1)
        if "BPA_F@1.0" in loop: loop["RQ_BPA"] = loop[["BPA_F@0.5", "BPA_F@1.0", "BPA_F@2.0"]].mean(1)
    return df, loop

def label3(df, good_rq=0.85, poor_rq=0.70):
    """Independent 3-class labels from the downstream outcome (NOT from CQI)."""
    y = np.where((df["PSR_print_ready"] == 1) & (df["RQ"] >= good_rq), 0, np.where(df["RQ"] < poor_rq, 2, 1))
    return y  # 0 Good, 1 Moderate, 2 Poor

# ---------------------------------------------------------------- fusion models
class EqualFusion:
    name = "CQI-equal"
    def fit(self, X, y, g=None): return self
    def predict(self, X, g=None): return X.mean(1)
    weights = None

class NNLSFusion:
    """Non-negative, sum-to-one weights learned from the downstream target (interpretable)."""
    name = "CQI-learned"
    def fit(self, X, y, g=None):
        A = np.c_[X, np.ones(len(X))]
        lb = np.r_[np.zeros(X.shape[1]), -np.inf]; ub = np.full(X.shape[1] + 1, np.inf)
        w = optimize.lsq_linear(A, y, bounds=(lb, ub)).x[:-1]
        self.weights = w / (w.sum() + 1e-12); return self
    def predict(self, X, g=None): return X @ self.weights

class GatedFusion:
    """Geometry-conditioned weights w(g)=softmax(Theta g): adaptive, learned, category-free."""
    name = "CQI-adaptive"
    def __init__(self, lam=1e-2): self.lam = lam
    def _w(self, G, th):
        Z = G @ th.reshape(G.shape[1], -1); Z -= Z.max(1, keepdims=True); E = np.exp(Z)
        return E / E.sum(1, keepdims=True)
    def fit(self, X, y, g):
        self.mu, self.sd = g.mean(0), g.std(0) + 1e-9
        G = np.c_[np.ones(len(g)), (g - self.mu) / self.sd]; k = X.shape[1]
        def loss(p):
            th, a, b = p[:-2], p[-2], p[-1]
            yh = a + b * (self._w(G, th) * X).sum(1)
            return np.mean((yh - y) ** 2) + self.lam * np.sum(th[k:] ** 2)
        p0 = np.r_[np.zeros(G.shape[1] * k), 0.0, 1.0]
        self.p = optimize.minimize(loss, p0, method="L-BFGS-B").x; return self
    def weights_for(self, g):
        G = np.c_[np.ones(len(g)), (g - self.mu) / self.sd]; return self._w(G, self.p[:-2])
    def predict(self, X, g): return (self.weights_for(g) * X).sum(1)

class HGB:
    name = "GBT (black-box)"
    def fit(self, X, y, g=None):
        self.m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(X, y); return self
    def predict(self, X, g=None): return self.m.predict(X)

# ---------------------------------------------------------------- evaluation helpers
def logistic5(x, b1, b2, b3, b4, b5): return b1 * (0.5 - 1 / (1 + np.exp(b2 * (x - b3)))) + b4 * x + b5

def plcc_rmse(p, y):
    try:
        popt, _ = optimize.curve_fit(logistic5, p, y, p0=[np.ptp(y), 1, np.mean(p), 0, np.mean(y)], maxfev=20000)
        ph = logistic5(p, *popt)
    except Exception:
        ph = np.polyval(np.polyfit(p, y, 1), p)
    return stats.pearsonr(ph, y)[0], float(np.sqrt(np.mean((ph - y) ** 2)))

def within_object_srcc(df, p, y="RQ"):
    v = []
    for _, g in df.assign(_p=p).groupby("object"):
        if g[y].nunique() > 2: v.append(stats.spearmanr(g["_p"], g[y])[0])
    return float(np.nanmean(v))

def boot_ci(df, p, y="RQ", n=1000, seed=0):
    rng = np.random.default_rng(seed); objs = df["object"].unique(); idx = df.groupby("object").indices
    vals = []
    for _ in range(n):
        s = np.concatenate([idx[o] for o in rng.choice(objs, len(objs))])
        vals.append(stats.spearmanr(p[s], df[y].values[s])[0])
    return np.percentile(vals, [2.5, 97.5])

def paired_boot_p(df, p1, p2, y="RQ", n=2000, seed=0):
    rng = np.random.default_rng(seed); objs = df["object"].unique(); idx = df.groupby("object").indices
    d = []
    for _ in range(n):
        s = np.concatenate([idx[o] for o in rng.choice(objs, len(objs))])
        d.append(stats.spearmanr(p1[s], df[y].values[s])[0] - stats.spearmanr(p2[s], df[y].values[s])[0])
    d = np.array(d); return float(np.mean(d)), float(2 * min((d <= 0).mean(), (d >= 0).mean()))

def best_thresholds(p, y3):
    qs = np.quantile(p, np.linspace(0.02, 0.98, 49)); best = (-1, None)
    for hi in qs:
        for lo in qs[qs < hi]:
            pr = np.where(p >= hi, 0, np.where(p < lo, 2, 1)); a = accuracy_score(y3, pr)
            if a > best[0]: best = (a, (lo, hi))
    return best[1]

# ---------------------------------------------------------------- main benchmark (cross-validated, object-disjoint)
def run_benchmark(df, out, n_splits=5):
    os.makedirs(out, exist_ok=True)
    tr_src = df[df.source != "stanford"].reset_index(drop=True)
    ext = df[df.source == "stanford"].reset_index(drop=True)
    y3_all = label3(tr_src)
    preds = {}; weights_log = []
    learners = {"CQI-equal": EqualFusion, "CQI-learned": NNLSFusion, "CQI-adaptive": GatedFusion, "GBT (black-box)": HGB}
    for nm in learners: preds[nm] = np.zeros(len(tr_src))
    th_pred = {nm: np.zeros(len(tr_src), int) for nm in learners}
    gkf = GroupKFold(n_splits=min(n_splits, tr_src.object.nunique()))
    for f, (a, b) in enumerate(gkf.split(tr_src, groups=tr_src.object)):
        A, B = tr_src.iloc[a], tr_src.iloc[b]
        for nm, L in learners.items():
            Xa = A[PROPOSED + (RAW if nm.startswith("GBT") else [])].values
            Xb = B[PROPOSED + (RAW if nm.startswith("GBT") else [])].values
            m = L().fit(Xa, A.RQ.values, A[GDESC].values)
            preds[nm][b] = m.predict(Xb, B[GDESC].values)
            pa = m.predict(Xa, A[GDESC].values); lo, hi = best_thresholds(pa, label3(A))
            th_pred[nm][b] = np.where(preds[nm][b] >= hi, 0, np.where(preds[nm][b] < lo, 2, 1))
            if nm == "CQI-learned": weights_log.append(dict(fold=f, **dict(zip(PROPOSED, m.weights))))
    # non-learned baselines (sign so that higher = better)
    base = {"CQI-v1 (previous manuscript)": tr_src["CQI_v1"].values,
            "FR Chamfer (needs reference)": -tr_src["FR_chamfer"].values,
            "FR Hausdorff (needs reference)": -tr_src["FR_hausdorff"].values,
            "Point count": tr_src["n_points"].values}
    for c in PROPOSED: base[f"single: {c}"] = tr_src[c].values
    for c in OLD: base[f"single (old): {c}"] = tr_src[c].values
    allp = {**preds, **base}
    rows = []
    pr_bin = tr_src["PSR_print_ready"].values
    for nm, p in allp.items():
        sr = stats.spearmanr(p, tr_src.RQ)[0]; kr = stats.kendalltau(p, tr_src.RQ)[0]
        pl, rm = plcc_rmse(p, tr_src.RQ.values); ci = boot_ci(tr_src, p)
        auc = roc_auc_score(pr_bin, p) if 0 < pr_bin.mean() < 1 else np.nan
        r = dict(method=nm, SRCC=sr, SRCC_lo=ci[0], SRCC_hi=ci[1], KRCC=kr, PLCC=pl, RMSE=rm,
                 within_obj_SRCC=within_object_srcc(tr_src, p), AUC_print_ready=auc)
        if nm in th_pred:
            r.update(Acc3=accuracy_score(y3_all, th_pred[nm]), MacroF1=f1_score(y3_all, th_pred[nm], average="macro"),
                     kappa_w=cohen_kappa_score(y3_all, th_pred[nm], weights="quadratic"))
        if "BPA_F@1.0" in tr_src: r["SRCC_vs_BPA_target"] = stats.spearmanr(p, tr_src.RQ_BPA)[0]
        rows.append(r)
    res = pd.DataFrame(rows).sort_values("SRCC", ascending=False); res.to_csv(f"{out}/T_benchmark_cv.csv", index=False)
    sig = []
    for a_, b_ in [("CQI-learned", "CQI-equal"), ("CQI-adaptive", "CQI-learned"), ("CQI-learned", "CQI-v1 (previous manuscript)"),
                   ("CQI-equal", "CQI-v1 (previous manuscript)"), ("GBT (black-box)", "CQI-learned")]:
        d, pv = paired_boot_p(tr_src, allp[a_], allp[b_]); sig.append(dict(A=a_, B=b_, dSRCC=d, p_boot=pv))
    pd.DataFrame(sig).to_csv(f"{out}/T_significance.csv", index=False)
    W = pd.DataFrame(weights_log); W.to_csv(f"{out}/T_learned_weights.csv", index=False)
    # external test: train on all non-Stanford, test on Stanford real scans
    ext_rows = []
    if len(ext):
        for nm, L in learners.items():
            cols = PROPOSED + (RAW if nm.startswith("GBT") else [])
            m = L().fit(tr_src[cols].values, tr_src.RQ.values, tr_src[GDESC].values)
            p = m.predict(ext[cols].values, ext[GDESC].values)
            ext_rows.append(dict(method=nm, SRCC=stats.spearmanr(p, ext.RQ)[0], PLCC=plcc_rmse(p, ext.RQ.values)[0],
                                 within_obj_SRCC=within_object_srcc(ext, p)))
        for nm, col, sgn in [("CQI-v1 (previous manuscript)", "CQI_v1", 1), ("FR Chamfer (needs reference)", "FR_chamfer", -1)]:
            p = sgn * ext[col].values
            ext_rows.append(dict(method=nm, SRCC=stats.spearmanr(p, ext.RQ)[0], PLCC=plcc_rmse(p, ext.RQ.values)[0],
                                 within_obj_SRCC=within_object_srcc(ext, p)))
        pd.DataFrame(ext_rows).to_csv(f"{out}/T_external_stanford.csv", index=False)
    # figures
    fig, ax = plt.subplots(figsize=(6.5, 3.6)); r2 = res.head(14).iloc[::-1]
    ax.barh(r2.method, r2.SRCC, xerr=[r2.SRCC - r2.SRCC_lo, r2.SRCC_hi - r2.SRCC], color="#4C72B0")
    ax.set_xlabel("SRCC with reconstruction-quality target (95% CI, object bootstrap)"); fig.tight_layout()
    fig.savefig(f"{out}/F_benchmark_srcc.png"); fig.savefig(f"{out}/F_benchmark_srcc.pdf"); plt.close(fig)
    fig, ax = plt.subplots(figsize=(3.4, 3.2)); p = preds["CQI-learned"]
    ax.scatter(p, tr_src.RQ, s=3, alpha=.35, c=tr_src.type.astype("category").cat.codes, cmap="tab10")
    ax.set_xlabel("CQI-learned (no reference)"); ax.set_ylabel("RQ (frozen Poisson pipeline)"); fig.tight_layout()
    fig.savefig(f"{out}/F_scatter_cqi_rq.png"); fig.savefig(f"{out}/F_scatter_cqi_rq.pdf"); plt.close(fig)
    if len(W):
        fig, ax = plt.subplots(figsize=(4.5, 2.6)); m_, s_ = W[PROPOSED].mean(), W[PROPOSED].std()
        ax.bar([c[2:] for c in PROPOSED], m_, yerr=s_, color="#55A868"); ax.set_ylabel("learned weight (mean±sd over folds)")
        plt.xticks(rotation=30); fig.tight_layout(); fig.savefig(f"{out}/F_weights.png"); fig.savefig(f"{out}/F_weights.pdf"); plt.close(fig)
    return res, tr_src.assign(**{f"pred_{k}": v for k, v in preds.items()})

# ---------------------------------------------------------------- metric specificity (single-factor experiments)
def specificity(df, out):
    sf = df[df.type.isin(SINGLE)]
    def mat(cols):
        M = pd.DataFrame(index=cols, columns=SINGLE, dtype=float)
        for t in SINGLE:
            g = sf[sf.type == t]
            for c in cols:   # mean per-object SRCC between metric and injected severity
                v = [stats.spearmanr(gg.severity, gg[c])[0] for _, gg in g.groupby("object") if gg[c].nunique() > 1]
                sgn = -1.0 if c in OLD else 1.0   # old metrics are quality scores -> flip to defect direction
                M.loc[c, t] = sgn * np.nanmean(v) if v else np.nan
        return M
    Mn, Mo = mat(RAW), mat(OLD + CONV)
    # effect size: median z-shift at max severity vs clean (IQR units) -> how LARGE, not just monotone
    ref = clean_reference(df); Mz = pd.DataFrame(index=RAW, columns=SINGLE, dtype=float)
    for t in SINGLE:
        g = sf[(sf.type == t) & (sf.severity == 1.0)]
        for c in RAW: Mz.loc[c, t] = float(np.median((g[c] - ref[c][0]) / ref[c][1])) if len(g) else np.nan
    Mz.to_csv(f"{out}/T_specificity_effectsize.csv")
    Mn.to_csv(f"{out}/T_specificity_proposed.csv"); Mo.to_csv(f"{out}/T_specificity_old_conventional.csv")
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, M, ttl in [(axs[0], Mn, "Proposed (raw, unclipped)"), (axs[1], Mo, "Previous / conventional")]:
        im = ax.imshow(M.values.astype(float), cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(SINGLE))); ax.set_xticklabels(SINGLE, rotation=35)
        ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                val = M.values[i, j]
                ax.text(j, i, "–" if np.isnan(val) else f"{val:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if abs(np.nan_to_num(val)) > 0.6 else "black")
        ax.set_title(ttl)
    fig.colorbar(im, ax=axs, shrink=.8, label="SRCC(defect magnitude, injected severity)")
    fig.savefig(f"{out}/F_specificity.png", bbox_inches="tight"); fig.savefig(f"{out}/F_specificity.pdf", bbox_inches="tight"); plt.close(fig)
    # noise calibration: estimated vs injected sigma (absolute mm)
    nz = df[df.type == "noise"]
    if len(nz) and "noise_sigma" in nz:
        fig, ax = plt.subplots(figsize=(3.3, 3.1))
        ax.loglog(nz.noise_sigma, nz.raw_noise_mm, ".", alpha=.4, ms=3, label="proposed (scale-extrapolated)")
        ax.loglog(nz.noise_sigma, nz.raw_noise_q24_mm, ".", alpha=.3, ms=3, label="single-scale quadric")
        lim = [nz.noise_sigma.min(), nz.noise_sigma.max()]; ax.plot(lim, lim, "k--", lw=.8, label="identity")
        ax.set_xlabel("injected σ (mm)"); ax.set_ylabel("estimated σ (mm)"); ax.legend(fontsize=6); fig.tight_layout()
        fig.savefig(f"{out}/F_noise_calibration.png"); fig.savefig(f"{out}/F_noise_calibration.pdf"); plt.close(fig)
        e = nz[nz.noise_sigma >= 0.05]
        pd.DataFrame([dict(median_abs_rel_err=float(np.median(np.abs(e.raw_noise_mm / e.noise_sigma - 1))),
                           SRCC=stats.spearmanr(e.raw_noise_mm, e.noise_sigma)[0])]).to_csv(f"{out}/T_noise_calibration.csv", index=False)
    return Mn, Mo, Mz

# ---------------------------------------------------------------- fault attribution (independent defect labels)
def clean_reference(df_train):
    c = df_train[df_train.type == "clean"]
    floor = dict(raw_outlier_frac=1e-3, raw_boundary_frac=1e-3)   # clean IQR can be 0
    return {k: (c[k].median(), max(c[k].quantile(.75) - c[k].quantile(.25), 0.1 * abs(c[k].median()),
                                   floor.get(k, 1e-6))) for k in RAW}

def zscores(df, ref):
    return pd.DataFrame({k: (df[k] - ref[k][0]) / ref[k][1] for k in RAW}, index=df.index)

def fault_attribution(df, out, z0=3.0, n_splits=5):
    gkf = GroupKFold(n_splits=min(n_splits, df.object.nunique())); res = []
    for a, b in gkf.split(df, groups=df.object):
        ref = clean_reference(df.iloc[a]); B = df.iloc[b]; Z = zscores(B, ref)
        sub = Z[list(DEFECT_OF)]
        pred = sub.idxmax(axis=1).map(DEFECT_OF)
        res.append(pd.DataFrame(dict(object=B.object, type=B.type, severity=B.severity, pred=pred,
                                     **{f"z_{k}": Z[k] for k in DEFECT_OF},
                                     **{f"sev_{t}": B.get(f"sev_{t}", np.nan) for t in ["noise", "sparsity", "nonuniform", "holes", "outliers"]})))
    R = pd.concat(res)
    single = R[R.type.isin(["noise", "sparsity", "nonuniform", "holes", "outliers"])]
    rows = []
    for s in sorted(single.severity.unique()):
        g = single[single.severity == s]
        rows.append(dict(severity=s, top1_acc=accuracy_score(g.type, g.pred), macroF1=f1_score(g.type, g.pred, average="macro")))
    pd.DataFrame(rows).to_csv(f"{out}/T_attribution_by_severity.csv", index=False)
    labs = ["noise", "sparsity", "nonuniform", "holes", "outliers"]
    g = single[single.severity >= 0.5]
    cm = confusion_matrix(g.type, g.pred, labels=labs, normalize="true")
    fig, ax = plt.subplots(figsize=(3.6, 3.2)); ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(5)); ax.set_xticklabels(labs, rotation=35); ax.set_yticks(range(5)); ax.set_yticklabels(labs)
    for i in range(5):
        for j in range(5): ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_xlabel("attributed defect"); ax.set_ylabel("injected defect (severity ≥ 0.5)"); fig.tight_layout()
    fig.savefig(f"{out}/F_attribution_cm.png"); fig.savefig(f"{out}/F_attribution_cm.pdf"); plt.close(fig)
    # mixed: multi-label detection (z > z0) vs truly active defects
    mx = R[R.type == "mixed"]; ml = []
    for k, t in DEFECT_OF.items():
        act = mx[f"sev_{t}"].notna() & (mx[f"sev_{t}"] >= 0.25); det = mx[f"z_{k}"] > z0
        tp = (act & det).sum(); ml.append(dict(defect=t, precision=tp / max(det.sum(), 1), recall=tp / max(act.sum(), 1), n_active=int(act.sum())))
    pd.DataFrame(ml).to_csv(f"{out}/T_attribution_mixed.csv", index=False)
    gh = R[R.type == "ghosting"].pred.value_counts(normalize=True)
    gh.to_csv(f"{out}/T_ghosting_unseen_attribution.csv")
    return R

# ---------------------------------------------------------------- closed-loop corrective actions
FIX_FOR = dict(raw_noise_mm="denoise", raw_outlier_frac="outliers", raw_density="uniformize")
def closed_loop(df, loop, out, z0=3.0, target="RQ"):
    if not len(loop) or target not in loop: return None
    tag = "" if target == "RQ" else "_" + target
    ref = clean_reference(df)                      # NOTE: for the paper use fold-wise refs (see notebook)
    base = df.set_index(["object", "variant"])
    rows = []
    for (o, v), g in loop.groupby(["object", "variant"]):
        if (o, v) not in base.index: continue
        b = base.loc[(o, v)]; z = {k: (b[k] - ref[k][0]) / ref[k][1] for k in RAW}
        rec = tuple(a for a in FIX_ORDER if any(FIX_FOR[k] == a and z[k] > z0 for k in FIX_FOR))
        recapture = any(z[k] > z0 for k in ["raw_spacing_mm", "raw_boundary_frac"])
        out_of = lambda combo: g[g.actions == "+".join(combo)][target].values[0] if len(combo) else b[target]
        singles = [out_of((a,)) for a in FIX_ORDER]
        rows.append(dict(object=o, variant=v, type=b.type, recommended="+".join(rec) or "none",
                         recapture_flag=recapture, RQ_none=b[target], RQ_policy=out_of(rec),
                         RQ_all=out_of(tuple(FIX_ORDER)), RQ_random_single=float(np.mean(singles)),
                         RQ_oracle=max([b[target]] + list(g[target].values))))
    C = pd.DataFrame(rows); C.to_csv(f"{out}/T_closed_loop_rows{tag}.csv", index=False)
    summ = []
    for col in ["RQ_policy", "RQ_all", "RQ_random_single", "RQ_oracle"]:
        d = C[col] - C.RQ_none
        p = stats.wilcoxon(C[col], C.RQ_none).pvalue if (d != 0).any() else np.nan
        summ.append(dict(strategy=col, mean_dRQ=d.mean(), improved=(d > 0.005).mean(), harmed=(d < -0.005).mean(), wilcoxon_p=p))
    PA = loop.assign(base=[base.loc[(o, v)][target] if (o, v) in base.index else np.nan for o, v in zip(loop.object, loop.variant)])
    PA = PA.assign(d=PA[target] - PA.base, vtype=PA.variant.str.split('_').str[0]).pivot_table(index='vtype', columns='actions', values='d', aggfunc='mean')
    PA.to_csv(f"{out}/T_closed_loop_per_action{tag}.csv")
    S = pd.DataFrame(summ); S.to_csv(f"{out}/T_closed_loop_summary{tag}.csv", index=False)
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.boxplot([C[c] - C.RQ_none for c in ["RQ_policy", "RQ_all", "RQ_random_single", "RQ_oracle"]],
               labels=["CQI policy", "apply all", "random single", "oracle"], showfliers=False)
    ax.axhline(0, color="k", lw=.6); ax.set_ylabel(f"Δ{target} after correction"); fig.tight_layout()
    fig.savefig(f"{out}/F_closed_loop{tag}.png"); fig.savefig(f"{out}/F_closed_loop{tag}.pdf"); plt.close(fig)
    return S

def runtime_table(df, out):
    T = df[["n_points", "t_metrics_s", "t_poisson_s"]].describe().T; T.to_csv(f"{out}/T_runtime.csv"); return T


In [ ]:
%%writefile prq_dtu.py
# =====================================================================
# prq_dtu.py - REAL-defect validation on DTU MVS 2014
# Inputs : real MVS point clouds (Campbell / Furukawa / Tola) = raw clouds
# Reference: structured-light scans (STL) + official observability masks
# CQI is FROZEN (weights learned on Thingi10K; nothing is fitted on DTU).
# =====================================================================
import os, re, numpy as np, pandas as pd, open3d as o3d
from scipy.io import loadmat
from scipy.spatial import cKDTree
import prq_core as C

BASE = "http://roboimagedata2.compute.dtu.dk/data/MVS"
URL = dict(sample=f"{BASE}/SampleSet.zip", points=f"{BASE}/Points.zip", mvs=f"{BASE}/Points_MVS.zip")
# 22 standard DTU evaluation scans (used by most MVS papers)
SCANS = [1, 4, 9, 10, 11, 12, 13, 15, 23, 24, 29, 32, 33, 34, 48, 49, 62, 75, 77, 110, 114, 118]
METHODS = ["camp", "furu", "tola"]
W_FROZEN = {"s_spacing": 0.2146, "s_density": 0.0, "s_noise": 0.0362, "s_holes": 0.3336,
            "s_outliers": 0.0319, "s_normals": 0.0772, "s_msnd": 0.3067}   # from Thingi10K CV
ROOT = "/tmp/dtu"

def _want(name, scans):
    b = os.path.basename(name)
    m = re.match(r"ObsMask(\d+)_10\.mat$", b) or re.match(r"Plane(\d+)\.mat$", b)
    if m: return int(m.group(1)) in scans
    m = re.match(r"stl(\d{3})_total\.ply$", b)
    if m: return int(m.group(1)) in scans
    m = re.match(r"(camp|furu|tola)(\d{3})(_l\d)?\.ply$", b)
    if m: return int(m.group(2)) in scans
    return False

def fetch(url, scans, out=ROOT):
    """Extract only the needed members. 1) HTTP range requests (remotezip, fast);
    2) fallback: stream the whole zip once and keep matching files (no disk for the zip)."""
    os.makedirs(out, exist_ok=True); got = []
    try:
        from remotezip import RemoteZip
        with RemoteZip(url, timeout=120) as z:
            for info in z.infolist():
                if not info.is_dir() and _want(info.filename, scans):
                    dst = os.path.join(out, os.path.basename(info.filename))
                    if not os.path.exists(dst):
                        with z.open(info) as src, open(dst + ".part", "wb") as f:
                            while True:
                                b = src.read(1 << 22)
                                if not b: break
                                f.write(b)
                        os.replace(dst + ".part", dst)
                    got.append(dst)
        print(f"[remotezip] {os.path.basename(url)}: {len(got)} files"); return got
    except Exception as e:
        print(f"[remotezip unavailable: {e!r}] -> streaming {os.path.basename(url)} (slower)")
    import urllib.request
    from stream_unzip import stream_unzip
    def chunks():
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=300) as r:
            while True:
                b = r.read(1 << 22)
                if not b: break
                yield b
    for name, size, parts in stream_unzip(chunks()):
        name = name.decode("utf-8", "replace")
        if _want(name, scans):
            dst = os.path.join(out, os.path.basename(name))
            with open(dst, "wb") as f:
                for p in parts: f.write(p)
            got.append(dst); print("  got", os.path.basename(name))
        else:
            for _ in parts: pass                     # must consume to advance the stream
    print(f"[stream] {os.path.basename(url)}: {len(got)} files"); return got

def download_all(scans=SCANS):
    for k in ["sample", "points", "mvs"]: fetch(URL[k], set(scans))

# ---------------------------------------------------------------- official DTU evaluation volume
class Volume:
    def __init__(self, scan):
        m = loadmat(f"{ROOT}/ObsMask{scan}_10.mat")
        self.BB = np.asarray(m["BB"], float); self.res = float(np.asarray(m["Res"]).ravel()[0])
        self.obs = np.asarray(m["ObsMask"]).astype(bool)
        self.P = np.asarray(loadmat(f"{ROOT}/Plane{scan}.mat")["P"], float).ravel()
    def above(self, X): return (np.c_[X, np.ones(len(X))] @ self.P) > 0
    def in_box(self, X): return np.all((X >= self.BB[0]) & (X <= self.BB[1]), 1)
    def in_mask(self, X):
        idx = np.round((X - self.BB[0]) / self.res).astype(int)
        ok = np.all((idx >= 0) & (idx < np.array(self.obs.shape)), 1)
        out = np.zeros(len(X), bool); out[ok] = self.obs[idx[ok, 0], idx[ok, 1], idx[ok, 2]]
        return out
    def eval_region(self, X): return self.in_mask(X) & self.above(X)

def _mvs_file(method, scan):
    cands = [f for f in os.listdir(ROOT) if re.match(rf"{method}{scan:03d}(_l\d)?\.ply$", f)]
    cands.sort(key=lambda f: (0 if "_l3" in f else 1, f))
    return os.path.join(ROOT, cands[0]) if cands else None

def _masked_eval(V, F, vol, c, s, Gn, treeG, prefix):
    o = {}
    if len(F) == 0:
        for t in C.CFG["TAUS"]: o[f"{prefix}F@{t}"] = 0.0
        return o
    R = C.sample_surface(V, F, C.CFG["N_EVAL"], seed=1)
    R = R[vol.eval_region(R / s + c)]                       # only the observed, above-table region
    if len(R) == 0:
        for t in C.CFG["TAUS"]: o[f"{prefix}F@{t}"] = 0.0
        return o
    a, _ = treeG.query(R, workers=-1); b, _ = cKDTree(R).query(Gn, workers=-1)
    for t in C.CFG["TAUS"]:
        p, r = (a < t).mean(), (b < t).mean(); o[f"{prefix}F@{t}"] = float(2 * p * r / (p + r + 1e-12))
    o[f"{prefix}chamfer"] = float(a.mean() + b.mean())
    return o

def process_scan(scan, out_dir, n_points=None, seed=42):
    os.makedirs(out_dir, exist_ok=True); f_out = f"{out_dir}/scan{scan:03d}.csv"
    if os.path.exists(f_out): return f_out
    rows = []
    try:
        vol = Volume(scan)
        G = np.asarray(o3d.io.read_point_cloud(f"{ROOT}/stl{scan:03d}_total.ply").points)
        G = G[vol.eval_region(G)]
        c = (G.max(0) + G.min(0)) / 2; s = C.CFG["PRINT_SIZE"] / (G.max(0) - G.min(0)).max()
        rng = np.random.default_rng(seed + scan)
        Gn = (G[rng.choice(len(G), min(len(G), C.CFG["N_EVAL"]), replace=False)] - c) * s
        treeG = cKDTree(Gn)
        for m in METHODS:
            f = _mvs_file(m, scan)
            if f is None: print("missing", m, scan); continue
            P = np.asarray(o3d.io.read_point_cloud(f).points)
            n_raw = len(P)
            P = P[vol.in_box(P) & vol.above(P)]              # object region of interest (same for all methods)
            n = n_points or C.CFG["N_POINTS"]
            if len(P) > n: P = P[rng.choice(len(P), n, replace=False)]   # same budget as training
            Pn = (P - c) * s
            r = dict(scan=scan, method=m, n_raw=n_raw, n_roi=len(P))
            r.update(C.proposed_metrics(Pn)); r.update(C.previous_metrics(Pn))
            r["CQI_learned"] = sum(W_FROZEN[k] * r[k] for k in W_FROZEN)
            r["CQI_equal"] = float(np.mean([r[k] for k in C.PROPOSED]))
            # raw-cloud DTU-style accuracy / completeness (privileged: needs reference)
            Pe = Pn[vol.eval_region(P)]
            if len(Pe):
                r["DTU_acc"] = float(treeG.query(Pe, workers=-1)[0].mean())
                r["DTU_comp"] = float(cKDTree(Pe).query(Gn, workers=-1)[0].mean())
                r["DTU_overall"] = 0.5 * (r["DTU_acc"] + r["DTU_comp"])
            r.update(_masked_eval(*C.reconstruct_poisson(Pn), vol, c, s, Gn, treeG, "PSR_"))
            if C.CFG["RUN_BPA"]:
                try: r.update(_masked_eval(*C.reconstruct_bpa(Pn), vol, c, s, Gn, treeG, "BPA_"))
                except Exception as e: print("bpa", scan, m, e)
            rows.append(r)
        pd.DataFrame(rows).to_csv(f_out, index=False)
    except Exception as e:
        with open(f"{out_dir}/scan{scan:03d}.err", "w") as fh: fh.write(repr(e))
        print("ERROR scan", scan, repr(e))
    return f_out


In [ ]:
%%writefile prq_extras.py
# =====================================================================
# prq_extras.py - closed-loop policies, noise-estimator baselines, DTU robustness
#  A  closed loop: fold-wise clean references, z-threshold sensitivity,
#     learned (logistic) defect detector
#  B  noise estimator vs baseline estimators + validation of the r^6 exponent
#  D  DTU robustness: point budget and reference-free crop/normalisation
# =====================================================================
import os, glob, zlib, numpy as np, pandas as pd
from scipy import stats
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
import prq_core as C

RAW = ["raw_spacing_mm", "raw_density", "raw_noise_mm", "raw_boundary_frac", "raw_outlier_frac", "raw_normal_dev", "raw_msnd"]
DEF = ["noise", "sparsity", "nonuniform", "holes", "outliers"]
Z_OF = dict(noise="raw_noise_mm", sparsity="raw_spacing_mm", nonuniform="raw_density", holes="raw_boundary_frac", outliers="raw_outlier_frac")
ACT = dict(noise="denoise", outliers="outliers", nonuniform="uniformize")      # sparsity/holes -> recapture
ORDER = ["outliers", "denoise", "uniformize"]

def clean_ref(d):
    c = d[d.type == "clean"]; fl = dict(raw_outlier_frac=1e-3, raw_boundary_frac=1e-3)
    return {k: (c[k].median(), max(c[k].quantile(.75) - c[k].quantile(.25), .1 * abs(c[k].median()), fl.get(k, 1e-6))) for k in RAW}
def zmat(X, ref): return pd.DataFrame({k: (X[k] - ref[k][0]) / ref[k][1] for k in RAW}, index=X.index)
def zfeat(X, ref): Z = zmat(X, ref).values; return np.sign(Z) * np.log1p(np.abs(Z))
def labels(X):
    return np.column_stack([((X.type == t) | ((X.type == "mixed") & (X.get(f"sev_{t}", pd.Series(np.nan, index=X.index)) >= .25))).values for t in DEF])

# ---------------------------------------------------------------- A closed loop
def closed_loop_policies(df, loop, out, thresholds=(2, 3, 4)):
    df = df.reset_index(drop=True); loop = loop.copy()
    for t in ["", "BPA_"]:
        if f"{t}F@1.0" not in loop and f"PSR_F@1.0" in loop and t == "": pass
    loop["RQ"] = loop[["PSR_F@0.5", "PSR_F@1.0", "PSR_F@2.0"]].mean(axis=1)
    if "BPA_F@1.0" in loop: loop["RQ_BPA"] = loop[["BPA_F@0.5", "BPA_F@1.0", "BPA_F@2.0"]].mean(axis=1)
    base = df.set_index(["object", "variant"])
    decisions = {}                                     # (object, variant, policy) -> action tuple
    for a, b in GroupKFold(5).split(df, groups=df.object):
        A, B = df.iloc[a], df.iloc[b]; ref = clean_ref(A); Z = zmat(B, ref)
        for th in thresholds:
            for i, r in B.iterrows():
                act = tuple(x for x in ORDER if any(ACT.get(d) == x and Z.loc[i, Z_OF[d]] > th for d in ACT))
                decisions[(r.object, r.variant, f"z>{th} (fold-wise)")] = act
        tr = A.type.isin(DEF + ["mixed", "clean"]); YA = labels(A[tr])
        P = np.column_stack([LogisticRegression(max_iter=3000).fit(zfeat(A[tr], ref), YA[:, j]).predict_proba(zfeat(B, ref))[:, 1] for j in range(5)])
        for k, (i, r) in enumerate(B.iterrows()):
            det = [DEF[j] for j in range(5) if P[k, j] > .5]
            decisions[(r.object, r.variant, "learned detector")] = tuple(x for x in ORDER if any(ACT.get(d) == x for d in det))
            decisions[(r.object, r.variant, "learned detector, SOR only")] = ("outliers",) if "outliers" in det else ()
    res = []
    for tgt in [c for c in ["RQ", "RQ_BPA"] if c in loop]:
        rows = []
        for (o, v), g in loop.groupby(["object", "variant"]):
            if (o, v) not in base.index: continue
            b0 = base.loc[(o, v)]; y0 = b0[tgt] if tgt in b0 else np.nan
            out_of = lambda combo: g[g.actions == "+".join(combo)][tgt].values[0] if combo else y0
            r = dict(object=o, variant=v, none=y0, all=out_of(tuple(ORDER)), oracle=max([y0] + list(g[tgt])),
                     random_single=float(np.mean([out_of((x,)) for x in ORDER])))
            for pol in sorted({k[2] for k in decisions}): r[pol] = out_of(decisions[(o, v, pol)])
            rows.append(r)
        R = pd.DataFrame(rows)
        for col in [c for c in R.columns if c not in ("object", "variant", "none")]:
            d = R[col] - R["none"]
            res.append(dict(target=tgt, strategy=col, mean_dRQ=d.mean(), improved=(d > .005).mean(), harmed=(d < -.005).mean(),
                            wilcoxon_p=stats.wilcoxon(R[col], R["none"]).pvalue if (d != 0).any() else np.nan))
    S = pd.DataFrame(res); S.to_csv(f"{out}/T_closed_loop_policies.csv", index=False); return S

# ---------------------------------------------------------------- B noise estimators
def _sig_quad(P, tree, q, k):
    _, ix = tree.query(P[q], k=k, workers=-1); u, w, h = C._local_uvh(P[ix])
    A = np.stack([u**2, u*w, w**2, u, w, np.ones_like(u)], -1)
    c = np.linalg.solve(np.einsum("mki,mkj->mij", A, A) + 1e-9*np.eye(6), np.einsum("mki,mk->mi", A, h)[..., None])[..., 0]
    r = h - np.einsum("mki,mi->mk", A, c)
    d, _ = tree.query(P[q], k=k, workers=-1)
    return float(np.median(np.sqrt((r**2).sum(1) / (k - 6)))), float(np.median(d[:, -1]))
def _plane(P, tree, q, k):
    _, ix = tree.query(P[q], k=k, workers=-1); _, _, h = C._local_uvh(P[ix]); return h
def estimators(P, seed=0):
    rng = np.random.default_rng(seed); tree = cKDTree(P); q = rng.choice(len(P), min(len(P), 5000), replace=False)
    o = {}
    s12, r12 = _sig_quad(P, tree, q, 12); s24, r24 = _sig_quad(P, tree, q, 24); s48, r48 = _sig_quad(P, tree, q, 48)
    lam = (r24 / r12) ** 6; o["proposed (2-scale, p=6)"] = float(np.sqrt(max((lam*s12**2 - s24**2) / (lam - 1), 0)))
    o["quadric k=12"], o["quadric k=24"] = s12, s24
    h12 = _plane(P, tree, q, 12)
    o["plane RMS k=12"] = float(np.median(np.sqrt((h12**2).sum(1) / (12 - 3))))
    o["plane MAD k=12"] = float(np.median(1.4826 * np.median(np.abs(h12 - np.median(h12, 1, keepdims=True)), 1)))
    o["min-over-scales plane"] = float(min(np.median(np.sqrt((_plane(P, tree, q, k)**2).sum(1) / (k - 3))) for k in (8, 12, 16, 24, 32)))
    # exponent: least-squares fit of sigma^2 = sn^2 + c r^p over three scales (p searched on a grid)
    rs, ss = np.array([r12, r24, r48]), np.array([s12, s24, s48])**2; best = (np.inf, 6, 0)
    for p in np.arange(2, 10.01, .25):
        X = np.c_[np.ones(3), rs**p]; coef, res_, *_ = np.linalg.lstsq(X, ss, rcond=None)
        e = float(((X @ coef - ss)**2).sum())
        if coef[0] >= 0 and coef[1] >= 0 and e < best[0]: best = (e, p, coef[0])
    o["fitted exponent p"] = best[1]; o["3-scale fitted-p estimate"] = float(np.sqrt(max(best[2], 0)))
    return o

def noise_baselines(objs, out, seed=42):
    rows = []
    for name, V, F in objs:
        V, F = C.normalize_mesh(V, F); P0 = C.sample_clean(V, F)
        rng = np.random.default_rng(zlib.crc32(f"{name}|nb|{seed}".encode()))
        sets = [("clean", 0.0, P0, 0.0)]
        for t in ["noise", "sparsity", "nonuniform"]:
            for s in C.LEVELS:
                P, meta = C.CORR[t](P0, s, rng); sets.append((t, s, P, meta.get("noise_sigma", 0.0)))
        for t, s, P, sig in sets:
            rows.append(dict(object=name, type=t, severity=s, true_sigma=sig, n=len(P), **estimators(P)))
    R = pd.DataFrame(rows); R.to_csv(f"{out}/noise_baselines_rows.csv", index=False)
    est = [c for c in R.columns if c not in ("object", "type", "severity", "true_sigma", "n", "fitted exponent p")]
    nz = R[(R.type == "noise") & (R.true_sigma >= .05)]; lo = R[(R.type == "noise") & (R.severity == 0)]
    sp = R[R.type == "sparsity"]; nu = R[R.type == "nonuniform"]
    T = []
    for e in est:
        T.append(dict(estimator=e, median_abs_rel_err=float(np.median(np.abs(nz[e] / nz.true_sigma - 1))),
                      bias_at_0p03mm=float(np.median(lo[e] - lo.true_sigma)),
                      SRCC_with_true_sigma=stats.spearmanr(R[R.type == "noise"][e], R[R.type == "noise"].true_sigma)[0],
                      confound_sparsity_SRCC=float(np.nanmean([stats.spearmanr(g.severity, g[e])[0] for _, g in sp.groupby("object")])),
                      confound_nonuniform_SRCC=float(np.nanmean([stats.spearmanr(g.severity, g[e])[0] for _, g in nu.groupby("object")]))))
    T = pd.DataFrame(T); T.to_csv(f"{out}/T_noise_baselines.csv", index=False)
    ex = R[R.type.isin(["clean", "noise"])].groupby("type")["fitted exponent p"].describe()
    ex.to_csv(f"{out}/T_noise_exponent.csv"); return T, ex

# ---------------------------------------------------------------- D DTU robustness
def dtu_variant(scan, method, budget, crop, out_dir, seed=42):
    import prq_dtu as Dm, open3d as o3d
    os.makedirs(out_dir, exist_ok=True); f_out = f"{out_dir}/scan{scan:03d}_{method}_{budget}_{crop}.csv"
    if os.path.exists(f_out): return f_out
    try:
        vol = Dm.Volume(scan)
        G = np.asarray(o3d.io.read_point_cloud(f"{Dm.ROOT}/stl{scan:03d}_total.ply").points); G = G[vol.eval_region(G)]
        P = np.asarray(o3d.io.read_point_cloud(Dm._mvs_file(method, scan)).points); n_raw = len(P)
        rng = np.random.default_rng(seed + scan)
        if crop == "reference":                      # main protocol: mask bounding box above table plane
            P = P[vol.in_box(P) & vol.above(P)]
            c = (G.max(0) + G.min(0)) / 2; s = C.CFG["PRINT_SIZE"] / (G.max(0) - G.min(0)).max()
        else:                                        # reference-free: cloud's own 1-99 % box (+10 %), own extent
            lo, hi = np.percentile(P, 1, 0), np.percentile(P, 99, 0); m = .1 * (hi - lo)
            P = P[np.all((P >= lo - m) & (P <= hi + m), 1)]
            c = (np.percentile(P, 99, 0) + np.percentile(P, 1, 0)) / 2
            s = C.CFG["PRINT_SIZE"] / (np.percentile(P, 99, 0) - np.percentile(P, 1, 0)).max()
        if len(P) > budget: P = P[rng.choice(len(P), budget, replace=False)]
        Pn = (P - c) * s
        Gn = (G[rng.choice(len(G), min(len(G), C.CFG["N_EVAL"]), replace=False)] - c) * s; treeG = cKDTree(Gn)
        r = dict(scan=scan, method=method, budget=budget, crop=crop, n_raw=n_raw, n_used=len(P))
        r.update(C.proposed_metrics(Pn)); r.update(C.previous_metrics(Pn))
        r["CQI_learned"] = sum(Dm.W_FROZEN[k] * r[k] for k in Dm.W_FROZEN)
        r["CQI_equal"] = float(np.mean([r[k] for k in C.PROPOSED]))
        r.update(Dm._masked_eval(*C.reconstruct_poisson(Pn), vol, c, s, Gn, treeG, "PSR_"))
        if budget <= 100000:
            try: r.update(Dm._masked_eval(*C.reconstruct_bpa(Pn), vol, c, s, Gn, treeG, "BPA_"))
            except Exception as e: print("bpa", scan, method, e)
        pd.DataFrame([r]).to_csv(f_out, index=False)
    except Exception as e:
        open(f_out.replace(".csv", ".err"), "w").write(repr(e)); print("ERR", scan, method, budget, crop, repr(e))
    return f_out

def dtu_robustness_table(out_dir, out):
    D = pd.concat([pd.read_csv(f) for f in glob.glob(f"{out_dir}/*.csv")], ignore_index=True)
    D["RQ"] = D[["PSR_F@0.5", "PSR_F@1.0", "PSR_F@2.0"]].mean(axis=1)
    if "BPA_F@1.0" in D: D["RQ_BPA"] = D[["BPA_F@0.5", "BPA_F@1.0", "BPA_F@2.0"]].mean(axis=1)
    def pw(g, a, b):
        ok = tot = 0
        for _, x in g.groupby("scan"):
            u, v = x[a].values, x[b].values
            for i in range(len(u)):
                for j in range(i + 1, len(u)):
                    if v[i] != v[j] and not np.isnan(v[i]) and not np.isnan(v[j]): tot += 1; ok += (u[i] - u[j]) * (v[i] - v[j]) > 0
        return ok / tot if tot else np.nan
    rows = []
    for (bud, crop), g in D.groupby(["budget", "crop"]):
        for pred in ["CQI_learned", "CQI_equal", "CQI_v1"]:
            r = dict(budget=bud, crop=crop, predictor=pred, n=len(g), spacing_mm_median=g.raw_spacing_mm.median(),
                     spacing_score_unique=g.s_spacing.round(4).nunique(),
                     SRCC_RQ_PSR=stats.spearmanr(g[pred], g.RQ)[0], pairwise_PSR=pw(g, pred, "RQ"))
            if "RQ_BPA" in g and g.RQ_BPA.notna().any():
                r["SRCC_RQ_BPA"] = stats.spearmanr(g[pred], g.RQ_BPA, nan_policy="omit")[0]; r["pairwise_BPA"] = pw(g, pred, "RQ_BPA")
            rows.append(r)
    T = pd.DataFrame(rows); T.to_csv(f"{out}/T_dtu_robustness.csv", index=False); D.to_csv(f"{out}/dtu_robustness_rows.csv", index=False)
    return T


In [ ]:
%%writefile prq_pointnet.py
# =====================================================================
# prq_pointnet.py - learned no-reference baseline (PointNet regressor)
# trained on the SAME regenerated clouds, SAME object-disjoint folds and
# SAME target (RQ) as CQI-learned; paired bootstrap comparison.
# =====================================================================
import zlib, numpy as np, pandas as pd
from scipy import stats, optimize
from sklearn.model_selection import GroupKFold
import prq_core as C

NPTS = 2048
PROPOSED = ["s_spacing", "s_density", "s_noise", "s_holes", "s_outliers", "s_normals", "s_msnd"]

def regenerate(obj_id, V, F, n_mixed=15, seed=42, npts=None):
    npts = npts or NPTS
    """Exactly the clouds of the main run (same seeds), each subsampled to NPTS points."""
    V, F = C.normalize_mesh(V, F); P0 = C.sample_clean(V, F)
    rng = np.random.default_rng(zlib.crc32(f"{obj_id}|{seed}".encode()))
    sub = np.random.default_rng(zlib.crc32(f"{obj_id}|sub".encode()))
    out = []
    for name, typ, sev, P, meta in C.make_variants(P0, rng, n_mixed):
        idx = sub.choice(len(P), npts, replace=len(P) < npts)
        out.append((obj_id, name, len(P), (P[idx] / 50.0).astype(np.float32)))   # 100 mm -> [-1, 1]
    return out

def fit_nnls(X, y):
    A = np.c_[X, np.ones(len(X))]
    w = optimize.lsq_linear(A, y, bounds=(np.r_[np.zeros(X.shape[1]), -np.inf], np.full(X.shape[1] + 1, np.inf))).x[:-1]
    return w / (w.sum() + 1e-12)

def evaluate(df, preds, out):
    """df: rows with object, RQ, source; preds: dict name -> out-of-fold prediction array."""
    def boot(p, y, n=1000):
        rng = np.random.default_rng(0); objs = df.object.unique(); idx = df.groupby("object").indices; v = []
        for _ in range(n):
            s = np.concatenate([idx[o] for o in rng.choice(objs, len(objs))]); v.append(stats.spearmanr(p[s], y[s])[0])
        return np.percentile(v, [2.5, 97.5]), np.array(v)
    y = df.RQ.values; rows = []; bs = {}
    for k, p in preds.items():
        ci, bs[k] = boot(p, y)
        w = np.nanmean([stats.spearmanr(g["_p"], g.RQ)[0] for _, g in df.assign(_p=p).groupby("object") if g.RQ.nunique() > 2])
        rows.append(dict(method=k, SRCC=stats.spearmanr(p, y)[0], lo=ci[0], hi=ci[1], within_object_SRCC=w))
    T = pd.DataFrame(rows)
    # paired bootstrap (same resamples because same seed / object order)
    if "PointNet" in bs and "CQI-learned" in bs:
        d = bs["PointNet"] - bs["CQI-learned"]
        T.attrs["paired"] = dict(dSRCC=float(d.mean()), p_boot=float(2 * min((d <= 0).mean(), (d >= 0).mean())))
    T.to_csv(f"{out}/T_pointnet_cv.csv", index=False); return T

def make_model(torch, n_extra=1):
    nn = torch.nn
    class PointNet(nn.Module):
        def __init__(s):
            super().__init__()
            s.f = nn.Sequential(nn.Conv1d(3, 64, 1), nn.BatchNorm1d(64), nn.ReLU(), nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                                nn.Conv1d(128, 512, 1), nn.BatchNorm1d(512), nn.ReLU())
            s.h = nn.Sequential(nn.Linear(512 + n_extra, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
        def forward(s, x, extra):                     # x: B x N x 3, extra: B x n_extra
            g = s.f(x.transpose(1, 2)).max(2)[0]
            return torch.sigmoid(s.h(torch.cat([g, extra], 1)))[:, 0]
    return PointNet()

def train_predict(torch, Xtr, ntr, ytr, Xte, nte, epochs=60, bs=64, lr=1e-3, seed=0):
    """ntr/nte: log point count (1-D) or extra features (2-D: log count + descriptor scores)."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(seed); np.random.seed(seed)
    ntr = np.asarray(ntr, np.float32).reshape(len(ntr), -1); nte = np.asarray(nte, np.float32).reshape(len(nte), -1)
    m = make_model(torch, ntr.shape[1]).to(dev); opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    Xtr_t = torch.tensor(Xtr); ntr_t = torch.tensor(ntr, dtype=torch.float32); ytr_t = torch.tensor(ytr, dtype=torch.float32)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm) - 1, bs):
            b = perm[i:i + bs]
            if len(b) < 2: continue
            x = Xtr_t[b].to(dev)
            R = torch.linalg.qr(torch.randn(len(b), 3, 3, device=dev))[0]          # random rotation augmentation
            x = torch.bmm(x, R)
            loss = torch.nn.functional.mse_loss(m(x, ntr_t[b].to(dev)), ytr_t[b].to(dev))
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Xte), 256):
            out.append(m(torch.tensor(Xte[i:i + 256]).to(dev), torch.tensor(nte[i:i + 256], dtype=torch.float32).to(dev)).cpu().numpy())
    return np.concatenate(out)

def run_cv(df, X, logn, train_fn, out):
    """df: Thingi rows aligned with X/logn. Same folds for every method."""
    df = df.reset_index(drop=True); preds = {"PointNet": np.zeros(len(df)), "CQI-learned": np.zeros(len(df)), "CQI-equal": df[PROPOSED].mean(1).values}
    for f, (a, b) in enumerate(GroupKFold(5).split(df, groups=df.object)):
        preds["PointNet"][b] = train_fn(X[a], logn[a], df.RQ.values[a], X[b], logn[b])
        w = fit_nnls(df[PROPOSED].values[a], df.RQ.values[a]); preds["CQI-learned"][b] = df[PROPOSED].values[b] @ w
        print(f"fold {f}: PointNet SRCC {stats.spearmanr(preds['PointNet'][b], df.RQ.values[b])[0]:.3f} | CQI-learned {stats.spearmanr(preds['CQI-learned'][b], df.RQ.values[b])[0]:.3f}")
    T = evaluate(df, preds, out); return T, preds


In [ ]:
%%writefile prq_usf.py
# Unsupported surface fraction (USF): no-reference completeness measure.
# Fraction of the frozen Poisson reconstruction farther than k x median spacing from any input point.
import os, zlib, numpy as np, pandas as pd
from scipy.spatial import cKDTree
import prq_core as C
KS = (2, 3, 5)
def usf(P, seed=1):
    V, F = C.reconstruct_poisson(P)
    if len(F) == 0: return {f"usf_k{k}": 1.0 for k in KS}
    R = C.sample_surface(V, F, 30000, seed=seed)
    tP = cKDTree(P); h = float(np.median(tP.query(P, k=7)[0][:, 1:].mean(1)))
    d = tP.query(R, workers=-1)[0]
    return {f"usf_k{k}": float((d > k * h).mean()) for k in KS}
def usf_object(obj_id, V, F, out_dir, n_mixed=15, seed=42):
    os.makedirs(out_dir, exist_ok=True); f = f"{out_dir}/{obj_id}.csv"
    if os.path.exists(f): return f
    V, F = C.normalize_mesh(V, F); P0 = C.sample_clean(V, F)
    rng = np.random.default_rng(zlib.crc32(f"{obj_id}|{seed}".encode()))
    rows = [dict(object=obj_id, variant=n, **usf(P)) for n, t, s, P, m in C.make_variants(P0, rng, n_mixed)]
    pd.DataFrame(rows).to_csv(f, index=False); return f

# ---------------------------------------------------------------- analysis
from scipy import stats, optimize
from sklearn.model_selection import GroupKFold
P7 = ["s_spacing", "s_density", "s_noise", "s_holes", "s_outliers", "s_normals", "s_msnd"]
def _nnls(X, y):
    A = np.c_[X, np.ones(len(X))]
    w = optimize.lsq_linear(A, y, bounds=(np.r_[np.zeros(X.shape[1]), -np.inf], np.full(X.shape[1] + 1, np.inf))).x[:-1]
    return w / (w.sum() + 1e-12)
def _boot(d, preds, y, n=2000):
    rng = np.random.default_rng(0); g = d.groupby(d.columns[0]).indices; keys = list(g); B = {k: [] for k in preds}
    for _ in range(n):
        s = np.concatenate([g[o] for o in rng.choice(keys, len(keys))])
        for k, p in preds.items(): B[k].append(stats.spearmanr(p[s], y[s])[0])
    return {k: np.array(v) for k, v in B.items()}
def analyze(rows, dtu, out, k=3, tau=0.05):
    os.makedirs(out, exist_ok=True); col = f"usf_k{k}"
    rows = rows.copy(); rows["s_usf"] = 2.0 ** (-rows[col] / tau); P8 = P7 + ["s_usf"]
    th = rows[rows.source != "stanford"].reset_index(drop=True); st = rows[rows.source == "stanford"].reset_index(drop=True)
    y = th.RQ.values; pred = {"CQI-learned (7)": np.zeros(len(th)), "CQI + USF (8)": np.zeros(len(th)), "USF alone": th.s_usf.values}
    W8 = []
    for a, b in GroupKFold(5).split(th, groups=th.object):
        w7 = _nnls(th[P7].values[a], y[a]); w8 = _nnls(th[P8].values[a], y[a]); W8.append(w8)
        pred["CQI-learned (7)"][b] = th[P7].values[b] @ w7; pred["CQI + USF (8)"][b] = th[P8].values[b] @ w8
    B = _boot(th[["object"]], pred, y)
    T = [dict(set="Thingi10K CV", method=m, SRCC=stats.spearmanr(p, y)[0], lo=np.percentile(B[m], 2.5), hi=np.percentile(B[m], 97.5)) for m, p in pred.items()]
    d = B["CQI + USF (8)"] - B["CQI-learned (7)"]
    T.append(dict(set="Thingi10K CV", method="(CQI + USF) minus CQI", SRCC=d.mean(), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
    w7 = _nnls(th[P7].values, y); w8 = _nnls(th[P8].values, y)
    for m, p in [("CQI-learned (7)", st[P7].values @ w7), ("CQI + USF (8)", st[P8].values @ w8), ("USF alone", st.s_usf.values)]:
        T.append(dict(set="Stanford external", method=m, SRCC=stats.spearmanr(p, st.RQ)[0]))
    # DTU: frozen weights from all Thingi10K clouds
    D = dtu.copy(); D["s_usf"] = 2.0 ** (-D[col] / tau)
    D["RQ"] = D[["PSR_F@0.5", "PSR_F@1.0", "PSR_F@2.0"]].mean(axis=1); D["RQ_BPA"] = D[["BPA_F@0.5", "BPA_F@1.0", "BPA_F@2.0"]].mean(axis=1)
    D["m7"] = D[P7].values @ w7; D["m8"] = D[P8].values @ w8; D["negcomp"] = -D.DTU_comp; D["negacc"] = -D.DTU_acc; D["negall"] = -D.DTU_overall
    def pw(a, b_):
        ok = tot = 0
        for _, g in D.groupby("scan"):
            u, v = g[a].values, g[b_].values
            for i in range(len(u)):
                for j in range(i + 1, len(u)):
                    if v[i] != v[j]: tot += 1; ok += (u[i] - u[j]) * (v[i] - v[j]) > 0
        return ok / tot
    for m, c in [("CQI-learned (7)", "m7"), ("CQI + USF (8)", "m8"), ("USF alone", "s_usf"), ("CQI-v1", "CQI_v1")]:
        T.append(dict(set="DTU frozen", method=m, SRCC=stats.spearmanr(D[c], D.RQ)[0], SRCC_BPA=stats.spearmanr(D[c], D.RQ_BPA)[0],
                      SRCC_negDTU=stats.spearmanr(D[c], D.negall)[0], pw_BPA=pw(c, "RQ_BPA"), pw_PSR=pw(c, "RQ"), pw_comp=pw(c, "negcomp"), pw_acc=pw(c, "negacc")))
    T = pd.DataFrame(T); T.to_csv(f"{out}/T_usf_k{k}.csv", index=False)
    # specificity of USF (median z-shift at max severity, IQR units of clean clouds)
    cl = rows[rows.type == "clean"][col]; m0, iqr = cl.median(), max(cl.quantile(.75) - cl.quantile(.25), 1e-3)
    S = rows[rows.severity == 1.0].groupby("type")[col].median().sub(m0).div(iqr).round(1)
    S.to_csv(f"{out}/T_usf_specificity_k{k}.csv")
    pd.DataFrame(dict(descriptor=P8, weight_mean=np.mean(W8, 0), weight_sd=np.std(W8, 0))).to_csv(f"{out}/T_usf_weights_k{k}.csv", index=False)
    D[["scan", "method", col, "s_usf", "m7", "m8", "RQ", "RQ_BPA", "DTU_acc", "DTU_comp"]].to_csv(f"{out}/dtu_usf_k{k}.csv", index=False)
    return T, S

def usf_dtu_scan(scan, out_dir):
    """Same region of interest, normalisation and random 20000-point subsample as prq_dtu.process_scan."""
    import prq_dtu as Dm, open3d as o3d
    os.makedirs(out_dir, exist_ok=True); f = f"{out_dir}/scan{scan:03d}.csv"
    if os.path.exists(f): return f
    vol = Dm.Volume(scan)
    G = np.asarray(o3d.io.read_point_cloud(f"{Dm.ROOT}/stl{scan:03d}_total.ply").points); G = G[vol.eval_region(G)]
    c = (G.max(0) + G.min(0)) / 2; s = C.CFG["PRINT_SIZE"] / (G.max(0) - G.min(0)).max()
    rng = np.random.default_rng(42 + scan); rng.choice(len(G), min(len(G), C.CFG["N_EVAL"]), replace=False)   # same draw order
    rows = []
    for m in Dm.METHODS:
        P = np.asarray(o3d.io.read_point_cloud(Dm._mvs_file(m, scan)).points); P = P[vol.in_box(P) & vol.above(P)]
        if len(P) > C.CFG["N_POINTS"]: P = P[rng.choice(len(P), C.CFG["N_POINTS"], replace=False)]
        rows.append(dict(scan=scan, method=m, n_check=len(P), **usf((P - c) * s)))
    pd.DataFrame(rows).to_csv(f, index=False); return f


In [ ]:
%%writefile dtu_qual.py
import numpy as np, pandas as pd, open3d as o3d, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import prq_core as C, prq_dtu as Dm
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":7,"savefig.dpi":600})
def dtu_figure(scan, out, elev=35, azim=-90, zoom=1.3, frontal=None, names=dict(camp="Campbell et al.",furu="Furukawa & Ponce",tola="Tola et al.")):
    vol=Dm.Volume(scan)
    G=np.asarray(o3d.io.read_point_cloud(f"{Dm.ROOT}/stl{scan:03d}_total.ply").points); G=G[vol.eval_region(G)]
    c=(G.max(0)+G.min(0))/2; s=C.CFG["PRINT_SIZE"]/(G.max(0)-G.min(0)).max(); rng=np.random.default_rng(42+scan)
    Gn=(G[rng.choice(len(G),min(len(G),C.CFG["N_EVAL"]),replace=False)]-c)*s; tG=cKDTree(Gn)
    cols=[("Reference (structured light)",Gn,None,None)]; rows=[]
    for m in Dm.METHODS:
        P=np.asarray(o3d.io.read_point_cloud(Dm._mvs_file(m,scan)).points); P=P[vol.in_box(P)&vol.above(P)]
        if len(P)>C.CFG["N_POINTS"]: P=P[rng.choice(len(P),C.CFG["N_POINTS"],replace=False)]
        Pn=(P-c)*s; met=C.proposed_metrics(Pn); cqi=sum(Dm.W_FROZEN[k]*met[k] for k in Dm.W_FROZEN)
        Vr,Fr=C.reconstruct_poisson(Pn); ev=Dm._masked_eval(Vr,Fr,vol,c,s,Gn,tG,"")
        rq=np.mean([ev[f"F@{t}"] for t in (0.5,1.0,2.0)])
        Pe=Pn[vol.eval_region(P)]; acc=tG.query(Pe)[0].mean(); comp=cKDTree(Pe).query(Gn)[0].mean()
        keep=vol.eval_region((Vr[Fr].mean(1))/s+c); Fk=Fr[keep]                      # show only the evaluated region
        cols.append((names[m],Pn,(Vr,Fk),f"CQI {cqi:.2f} | RQ {rq:.2f}\nacc {acc:.2f} | comp {comp:.2f}"))
        rows.append(dict(scan=scan,method=m,CQI=cqi,RQ=rq,acc=acc,comp=comp))
    if frontal is not None:      # rotate so the thinnest extent of the reference faces the camera (y axis)
        w,v=np.linalg.eigh(np.cov(Gn.T)); R=np.c_[v[:,2],v[:,0]*(-1 if frontal else 1),v[:,1]]
        rot=lambda X: X@R
        cols=[(l,rot(P),None if m is None else (rot(m[0]),m[1]),t) for l,P,m,t in cols]
        elev,azim=8,-90
    f=plt.figure(figsize=(7.2,3.6)); lim=np.abs(cols[0][1]).max()*0.75
    for j,(lab,P,mesh,txt) in enumerate(cols):
        ax=f.add_subplot(2,4,j+1,projection="3d")
        ax.scatter(P[:,0],P[:,1],P[:,2],s=.04,c=P[:,2],cmap="viridis",linewidths=0,rasterized=True); ax.set_title(lab,fontsize=7,pad=0)
        ax2=f.add_subplot(2,4,j+5,projection="3d")
        if mesh is not None and len(mesh[1]):
            mm=o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(mesh[0]),o3d.utility.Vector3iVector(mesh[1]))
            mm.remove_unreferenced_vertices()
            if len(mm.triangles)>20000: mm=mm.simplify_quadric_decimation(20000)
            V=np.asarray(mm.vertices); Fd=np.asarray(mm.triangles)
            ax2.plot_trisurf(V[:,0],V[:,1],V[:,2],triangles=Fd,color="#c9d6e8",linewidth=0,shade=True,antialiased=False,rasterized=True)
            ax2.set_title(txt,fontsize=6.2,pad=0)
        else: ax2.set_visible(False)
        for a in (ax,ax2):
            a.view_init(elev=elev,azim=azim); a.set_axis_off(); a.set_box_aspect(None,zoom=zoom)
            a.set_xlim(-lim,lim); a.set_ylim(-lim,lim); a.set_zlim(-lim,lim)
    f.subplots_adjust(left=.03,right=1,top=.93,bottom=0,wspace=-.05,hspace=.1)
    f.text(0,.62,"Point cloud",rotation=90,fontsize=7,fontweight="bold"); f.text(0,.08,"Reconstruction",rotation=90,fontsize=7,fontweight="bold")
    tag="" if frontal is None else f"_frontal{int(frontal)}"
    f.savefig(f"{out}/Fig_dtu_examples_scan{scan}{tag}.png",bbox_inches="tight"); f.savefig(f"{out}/Fig_dtu_examples_scan{scan}{tag}.tif",bbox_inches="tight",pil_kwargs={"compression":"tiff_lzw"})
    plt.close(f); T=pd.DataFrame(rows); T.to_csv(f"{out}/dtu_examples_scan{scan}.csv",index=False); return T


In [ ]:
%%writefile figlib.py
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix
RAW=["raw_spacing_mm","raw_density","raw_noise_mm","raw_boundary_frac","raw_outlier_frac","raw_normal_dev","raw_msnd"]
DEFECT_OF=dict(raw_noise_mm="noise",raw_spacing_mm="sparsity",raw_density="nonuniform",raw_boundary_frac="holes",raw_outlier_frac="outliers")
def clean_ref(d):
    c=d[d.type=="clean"]; floor=dict(raw_outlier_frac=1e-3,raw_boundary_frac=1e-3)
    return {k:(c[k].median(),max(c[k].quantile(.75)-c[k].quantile(.25),0.1*abs(c[k].median()),floor.get(k,1e-6))) for k in RAW}
def attribution_cm(df):
    tr=df.reset_index(drop=True); out=[]
    for a,b in GroupKFold(n_splits=5).split(tr,groups=tr.object):
        ref=clean_ref(tr.iloc[a]); B=tr.iloc[b]
        Z=pd.DataFrame({k:(B[k]-ref[k][0])/ref[k][1] for k in DEFECT_OF},index=B.index)
        out.append(pd.DataFrame(dict(type=B.type,severity=B.severity,pred=Z.idxmax(axis=1).map(DEFECT_OF))))
    R=pd.concat(out); labs=["noise","sparsity","nonuniform","holes","outliers"]
    g=R[R.type.isin(labs)&(R.severity>=0.5)]
    return confusion_matrix(g.type,g.pred,labels=labs,normalize="true")

def _zf(X, ref):
    Z = np.column_stack([(X[k]-ref[k][0])/ref[k][1] for k in RAW]); return np.sign(Z)*np.log1p(np.abs(Z))
def learned_attribution(df):
    """Multinomial logistic regression on signed-log z-scores of all 7 descriptors (training folds only)."""
    from sklearn.linear_model import LogisticRegression
    labs = ["noise","sparsity","nonuniform","holes","outliers"]; tr = df.reset_index(drop=True); out = []
    for a, b in GroupKFold(n_splits=5).split(tr, groups=tr.object):
        A, B = tr.iloc[a], tr.iloc[b]; ref = clean_ref(A); s = A.type.isin(labs)
        m = LogisticRegression(max_iter=2000).fit(_zf(A[s], ref), A.type[s])
        Zr = pd.DataFrame({k:(B[k]-ref[k][0])/ref[k][1] for k in DEFECT_OF}, index=B.index)
        out.append(pd.DataFrame(dict(type=B.type, severity=B.severity, pred=m.predict(_zf(B, ref)), rule=Zr.idxmax(axis=1).map(DEFECT_OF))))
    R = pd.concat(out); g = R[R.type.isin(labs)]
    sev = g.groupby("severity").apply(lambda x: pd.Series(dict(rule=(x.rule==x.type).mean(), learned=(x.pred==x.type).mean())))
    h = g[g.severity >= .5]
    return sev, confusion_matrix(h.type, h.pred, labels=labs, normalize="true"), R


In [ ]:
%%writefile prq_dtu_ext.py
# Extended DTU evaluation: every scan that has all three MVS clouds, a reference scan and an observability mask.
import re, glob, numpy as np, pandas as pd
from scipy import stats
import prq_dtu as Dm
def available_scans():
    from remotezip import RemoteZip
    names = {}
    for k in ("sample", "points", "mvs"):
        with RemoteZip(Dm.URL[k], timeout=120) as z: names[k] = [i.filename for i in z.infolist()]
    mask = {int(m.group(1)) for n in names["sample"] if (m := re.search(r"ObsMask(\d+)_10\.mat$", n))}
    plane = {int(m.group(1)) for n in names["sample"] if (m := re.search(r"Plane(\d+)\.mat$", n))}
    stl = {int(m.group(1)) for n in names["points"] if (m := re.search(r"stl(\d{3})_total\.ply$", n))}
    mvs = {}
    for n in names["mvs"]:
        m = re.search(r"(camp|furu|tola)(\d{3})_l3\.ply$", n)
        if m: mvs.setdefault(int(m.group(2)), set()).add(m.group(1))
    full = {s for s, ms in mvs.items() if ms == {"camp", "furu", "tola"}}
    scans = sorted(mask & plane & stl & full)
    print(f"masks {len(mask)} | reference scans {len(stl)} | scans with all 3 MVS clouds {len(full)} -> usable {len(scans)}")
    return scans
def analyze(D, out, n_boot=2000):
    D = D.copy(); D["RQ"] = D[["PSR_F@0.5", "PSR_F@1.0", "PSR_F@2.0"]].mean(axis=1)
    D["RQ_BPA"] = D[["BPA_F@0.5", "BPA_F@1.0", "BPA_F@2.0"]].mean(axis=1)
    D["negacc"], D["negcomp"], D["negall"] = -D.DTU_acc, -D.DTU_comp, -D.DTU_overall
    D["subset"] = np.where(D.scan.isin(Dm.SCANS), "standard 22", "additional")
    def pw(g, a, b):
        ok = tot = 0
        for _, x in g.groupby("scan"):
            u, v = x[a].values, x[b].values
            for i in range(len(u)):
                for j in range(i + 1, len(u)):
                    if v[i] != v[j]: tot += 1; ok += (u[i] - u[j]) * (v[i] - v[j]) > 0
        return ok / tot if tot else np.nan
    def boot(g, p, y):
        rng = np.random.default_rng(0); sc = g.scan.unique(); idx = g.reset_index(drop=True).groupby("scan").indices; v = []
        G = g.reset_index(drop=True)
        for _ in range(n_boot):
            s = np.concatenate([idx[k] for k in rng.choice(sc, len(sc))]); v.append(stats.spearmanr(G[p].values[s], G[y].values[s])[0])
        return np.array(v)
    rows = []
    for name, g in [("all scans", D), ("standard 22", D[D.subset == "standard 22"]), ("additional", D[D.subset == "additional"])]:
        if g.scan.nunique() < 3: continue
        for pred in ["CQI_learned", "CQI_equal", "CQI_v1"]:
            r = dict(subset=name, scenes=g.scan.nunique(), clouds=len(g), predictor=pred)
            for y in ["RQ", "RQ_BPA", "negall"]:
                b = boot(g, pred, y); r[f"SRCC_{y}"] = stats.spearmanr(g[pred], g[y])[0]; r[f"lo_{y}"], r[f"hi_{y}"] = np.percentile(b, [2.5, 97.5])
            for y in ["RQ", "RQ_BPA", "negacc", "negcomp"]: r[f"pairwise_{y}"] = pw(g, pred, y)
            rows.append(r)
        bl, bv = boot(g, "CQI_learned", "RQ"), boot(g, "CQI_v1", "RQ"); d = bl - bv
        rows.append(dict(subset=name, predictor="CQI_learned minus CQI_v1 (Poisson RQ)", SRCC_RQ=d.mean(), lo_RQ=np.percentile(d, 2.5), hi_RQ=np.percentile(d, 97.5), p=2 * min((d <= 0).mean(), (d >= 0).mean())))
        S = g.groupby("scan")[["CQI_learned", "RQ"]].mean(); rows.append(dict(subset=name, predictor="scene-level CQI_learned", SRCC_RQ=stats.spearmanr(S.CQI_learned, S.RQ)[0]))
    T = pd.DataFrame(rows); T.to_csv(f"{out}/T_dtu_extended.csv", index=False); D.to_csv(f"{out}/dtu_extended_rows.csv", index=False)
    return T


In [ ]:
import prq_core as C, prq_run as Rn, prq_analysis as A, prq_dtu as Dm, prq_dtu_ext as E, prq_extras as X, prq_pointnet as PN, prq_usf as U, dtu_qual as Q
from joblib import Parallel, delayed
from scipy import stats
NCPU = os.cpu_count()

## Stage 1 · Controlled benchmark (resumable)

In [ ]:
if RUN['benchmark']:
    st = Rn.load_stanford(); ents = Rn.thingi_ids(N_THINGI, seed=SEED)
    pd.DataFrame([{k: e[k] for k in ('file_id','thing_id','license')} for e in ents]).to_csv(f'{W}/thingi_selection.csv', index=False)
    def job_thingi(i, e):
        V, F = Rn.load_thingi(e)
        return Rn.process_object(f"thingi_{e['file_id']}", V, F, 'thingi', D_ROWS, N_MIXED, closed_loop=(i < CLOSED_LOOP_N), seed=SEED)
    t = time.time()
    # split across two accounts; original positions kept, so the same 40 objects get the closed-loop analysis
    thingi_jobs = list(enumerate(ents)) if SPLIT is None else list(enumerate(ents))[SPLIT::2]
    stanford_jobs = list(st.items()) if SPLIT in (None, 0) else []
    print(f'split {SPLIT}: {len(stanford_jobs)} Stanford + {len(thingi_jobs)} Thingi10K objects')
    Parallel(n_jobs=NCPU, verbose=5)([delayed(Rn.process_object)(f'stanford_{k}', V, F, 'stanford', D_ROWS, N_MIXED, True, SEED) for k, (V, F) in stanford_jobs] +
                                     [delayed(job_thingi)(i, e) for i, e in thingi_jobs])
    print(f'done in {(time.time()-t)/3600:.2f} h; errors:', glob.glob(D_ROWS + '/*.err'))

## Stage 2 · Benchmark analysis

In [ ]:
if RUN['analysis']:
    df, loop = A.load(D_ROWS); df = df[df.source != 'smoke']
    print(df.groupby('source').object.nunique().to_dict(), len(df), 'clouds')
    res, preds = A.run_benchmark(df, D_RES); display(res.round(3))
    display(pd.read_csv(D_RES + '/T_significance.csv').round(4)); display(pd.read_csv(D_RES + '/T_external_stanford.csv').round(3))
    Mn, Mo, Mz = A.specificity(df, D_RES); display(Mz.round(1))
    A.fault_attribution(df, D_RES); display(pd.read_csv(D_RES + '/T_attribution_by_severity.csv').round(3))
    display(A.runtime_table(df, D_RES))
    df.to_csv(D_RES + '/all_rows.csv', index=False); preds.to_csv(D_RES + '/cv_predictions.csv', index=False)
    # descriptor-parameter sensitivity (12 objects, descriptors only)
    st = Rn.load_stanford(); ents = Rn.thingi_ids(N_THINGI, seed=SEED)
    sens_objs = [(k, *v) for k, v in list(st.items())[:2]] + [(f"t{e['file_id']}", *Rn.load_thingi(e)) for e in ents[:10]]
    settings = {'K_NOISE': [(8, 16), (12, 24), (16, 32)], 'K_BND': [12, 16, 24], 'BND_THR': [np.pi/2, 2*np.pi/3, 3*np.pi/4]}
    base = {k: C.CFG[k] for k in settings}; rows = []
    for o, V, F in sens_objs:
        V, F = C.normalize_mesh(V, F); P0 = C.sample_clean(V, F); rng = np.random.default_rng(1)
        vars_ = [(t, s, C.CORR[t](P0, s, rng)[0]) for t in ['noise', 'sparsity', 'holes'] for s in C.LEVELS]
        for key, vals in settings.items():
            for val in vals:
                C.CFG.update(base); C.CFG[key] = val
                for t, s, P in vars_:
                    m = C.proposed_metrics(P); rows.append(dict(object=o, param=key, value=str(val), type=t, severity=s, **{k: m[k] for k in C.RAW}))
    C.CFG.update(base); SE = pd.DataFrame(rows); out = []
    for (p, v), g in SE.groupby(['param', 'value']):
        r = dict(param=p, value=v)
        for t, col in [('noise','raw_noise_mm'), ('sparsity','raw_noise_mm'), ('holes','raw_boundary_frac'), ('sparsity','raw_boundary_frac')]:
            gg = g[g.type == t]; r[f'{col}|{t}'] = np.nanmean([stats.spearmanr(x.severity, x[col])[0] for _, x in gg.groupby('object')])
        out.append(r)
    pd.DataFrame(out).to_csv(D_RES + '/T_sensitivity.csv', index=False)
    # runtime scaling and MSND-curvature validation
    import tracemalloc, trimesh
    V, F = C.normalize_mesh(*st['Armadillo']); rt = []
    for n in [5000, 10000, 20000, 50000, 100000, 200000]:
        P = C.sample_surface(V, F, n); tracemalloc.start(); t = time.perf_counter(); C.proposed_metrics(P)
        rt.append(dict(n_points=n, metrics_s=time.perf_counter() - t, peak_MB=tracemalloc.get_traced_memory()[1] / 2**20)); tracemalloc.stop()
    pd.DataFrame(rt).to_csv(D_RES + '/T_runtime_scaling.csv', index=False)
    ms = []
    for o, V, F in sens_objs:
        V, F = C.normalize_mesh(V, F); P = C.sample_clean(V, F); _, pw = C.proposed_metrics(P, return_pointwise=True)
        Qp = P[pw['q']][:1500]; H = trimesh.curvature.discrete_mean_curvature_measure(trimesh.Trimesh(V, F), Qp, 2.0) / (np.pi * 4.0)
        ms.append(dict(object=o, SRCC_MSND_vs_absH=stats.spearmanr(pw['msnd'][:1500], np.abs(H))[0]))
    pd.DataFrame(ms).to_csv(D_RES + '/T_msnd_curvature.csv', index=False)

## Stage 3 · Real-data validation on DTU MVS 2014

In [ ]:
if RUN['dtu']:
    t = time.time(); Dm.download_all(Dm.SCANS); print(f'download {(time.time()-t)/60:.1f} min')
    Parallel(n_jobs=NCPU, verbose=5)(delayed(Dm.process_scan)(s, D_DTU) for s in Dm.SCANS)
    DD = pd.concat([pd.read_csv(f) for f in glob.glob(D_DTU + '/*.csv')], ignore_index=True)
    DD['RQ'] = DD[['PSR_F@0.5','PSR_F@1.0','PSR_F@2.0']].mean(axis=1); DD['RQ_BPA'] = DD[['BPA_F@0.5','BPA_F@1.0','BPA_F@2.0']].mean(axis=1)
    DD.to_csv(D_DTU_RES + '/dtu_all_rows.csv', index=False)
    rows = []
    for name, col, sgn in [('CQI-learned','CQI_learned',1), ('CQI-equal','CQI_equal',1), ('CQI-v1','CQI_v1',1)]:
        p = sgn * DD[col]
        rows.append(dict(predictor=name, SRCC_RQ=stats.spearmanr(p, DD.RQ)[0], SRCC_RQ_BPA=stats.spearmanr(p, DD.RQ_BPA)[0],
                         SRCC_negDTU=stats.spearmanr(p, -DD.DTU_overall)[0]))
    T = pd.DataFrame(rows); T.to_csv(D_DTU_RES + '/T_dtu_benchmark.csv', index=False); display(T.round(3))

## Stage 3b · Extended DTU validation (all 80 scenes, 240 clouds)

In [ ]:
if RUN['dtu_extended']:
    # all DTU scans with reconstructions by all three MVS methods, a reference scan and an observability mask
    SCANS_ALL = E.available_scans(); print(len(SCANS_ALL), 'scans:', SCANS_ALL)
    for k in ['sample', 'points', 'mvs']: Dm.fetch(Dm.URL[k], set(SCANS_ALL))
    Parallel(n_jobs=NCPU, verbose=5)(delayed(Dm.process_scan)(s, D_DTUX + '/rows') for s in SCANS_ALL)
    DX = pd.concat([pd.read_csv(f) for f in glob.glob(D_DTUX + '/rows/*.csv')], ignore_index=True)
    print(DX.scan.nunique(), 'scenes,', len(DX), 'clouds'); display(E.analyze(DX, D_DTUX).round(3))

## Stage 4 · Correction policies, noise-estimator baselines, DTU robustness

In [ ]:
if RUN['extras']:
    df = pd.read_csv(find('all_rows.csv')); _, loop = A.load(D_ROWS)
    display(X.closed_loop_policies(df, loop, D_EXTRA).round(3))
    st = Rn.load_stanford(); ents = Rn.thingi_ids(N_THINGI, seed=SEED)
    objs = [(f'stanford_{k}', V, F) for k, (V, F) in st.items()] + [(f"thingi_{e['file_id']}", *Rn.load_thingi(e)) for e in ents[:36]]
    for i in range(len(objs)): os.makedirs(f'{D_EXTRA}/nb_{i}', exist_ok=True)
    Parallel(n_jobs=NCPU, verbose=2)(delayed(X.noise_baselines)([o], f'{D_EXTRA}/nb_{i}') for i, o in enumerate(objs))
    R = pd.concat([pd.read_csv(f) for f in glob.glob(f'{D_EXTRA}/nb_*/noise_baselines_rows.csv')]); R.to_csv(D_EXTRA + '/noise_baselines_rows.csv', index=False)
    est = [c for c in R.columns if c not in ('object','type','severity','true_sigma','n','fitted exponent p')]
    nz = R[(R.type=='noise') & (R.true_sigma>=.05)]; lo = R[(R.type=='noise') & (R.severity==0)]
    T = pd.DataFrame([dict(estimator=e, median_abs_rel_err=np.median(np.abs(nz[e]/nz.true_sigma-1)), bias_at_0p03mm=np.median(lo[e]-lo.true_sigma),
        confound_sparsity_SRCC=np.nanmean([stats.spearmanr(g.severity, g[e])[0] for _, g in R[R.type=='sparsity'].groupby('object')]),
        confound_nonuniform_SRCC=np.nanmean([stats.spearmanr(g.severity, g[e])[0] for _, g in R[R.type=='nonuniform'].groupby('object')])) for e in est])
    T.to_csv(D_EXTRA + '/T_noise_baselines.csv', index=False); display(T.round(3))
    R[R.type.isin(['clean','noise'])].groupby(['type','severity'])['fitted exponent p'].describe().to_csv(D_EXTRA + '/T_noise_exponent.csv')
    Dm.download_all(Dm.SCANS)
    jobs = [(s, m, b, 'reference') for s in Dm.SCANS for m in Dm.METHODS for b in (20000, 100000, 500000)] + [(s, m, 20000, 'self') for s in Dm.SCANS for m in Dm.METHODS]
    Parallel(n_jobs=NCPU, verbose=5)(delayed(X.dtu_variant)(*j, D_EXTRA + '/dtu_rob') for j in jobs)
    display(X.dtu_robustness_table(D_EXTRA + '/dtu_rob', D_EXTRA).round(3))

## Stage 5 · Learned baselines (GPU): PointNet 2048 / 8192 points and hybrid

In [ ]:
if RUN['pointnet']:
    import torch; print('CUDA:', torch.cuda.is_available())
    df = pd.read_csv(find('all_rows.csv')); df = df[df.source != 'smoke']
    st = Rn.load_stanford(); ents = Rn.thingi_ids(N_THINGI, seed=SEED)
    def clouds(npts):
        G = Parallel(n_jobs=NCPU)([delayed(PN.regenerate)(f'stanford_{k}', V, F, npts=npts) for k, (V, F) in st.items()] +
                                  [delayed(lambda e: PN.regenerate(f"thingi_{e['file_id']}", *Rn.load_thingi(e), npts=npts))(e) for e in ents])
        key = {(o, v): (n, Xc) for g in G for o, v, n, Xc in g}
        d = df[[(o, v) in key for o, v in zip(df.object, df.variant)]].reset_index(drop=True)
        n = np.array([key[(o, v)][0] for o, v in zip(d.object, d.variant)]); print('point counts identical:', (n == d.n_points.values).mean())
        return d, np.stack([key[(o, v)][1] for o, v in zip(d.object, d.variant)]), np.log10(n).astype(np.float32)
    # (a) PointNet with 2048 points
    d, Xp, ln = clouds(2048); th = (d.source != 'stanford').values
    train = lambda Xa, na, ya, Xb, nb: PN.train_predict(torch, Xa, na, ya, Xb, nb, epochs=60)
    T, _ = PN.run_cv(d[th], Xp[th], ln[th], train, D_PN); display(T.round(3))
    del Xp
    # (b) PointNet with 8192 points and hybrid (PointNet + 7 descriptor scores)
    from sklearn.model_selection import GroupKFold
    d, Xp, ln = clouds(8192); th = (d.source != 'stanford').values; E = np.c_[ln, d[PN.PROPOSED].values].astype(np.float32)
    dt = d[th].reset_index(drop=True); Xt, lt, Et, y = Xp[th], ln[th], E[th], dt.RQ.values
    P = {k: np.zeros(len(dt)) for k in ['PointNet-8192', 'Hybrid (PointNet + descriptors)', 'CQI-learned']}
    for a, b in GroupKFold(5).split(dt, groups=dt.object):
        P['PointNet-8192'][b] = PN.train_predict(torch, Xt[a], lt[a], y[a], Xt[b], lt[b], epochs=40)
        P['Hybrid (PointNet + descriptors)'][b] = PN.train_predict(torch, Xt[a], Et[a], y[a], Xt[b], Et[b], epochs=40)
        P['CQI-learned'][b] = dt[PN.PROPOSED].values[b] @ PN.fit_nnls(dt[PN.PROPOSED].values[a], y[a])
    rng = np.random.default_rng(0); objs = dt.object.unique(); idx = dt.groupby('object').indices; B = {k: [] for k in P}
    for _ in range(2000):
        s = np.concatenate([idx[o] for o in rng.choice(objs, len(objs))])
        for k in P: B[k].append(stats.spearmanr(P[k][s], y[s])[0])
    B = {k: np.array(v) for k, v in B.items()}
    rows = [dict(method=k, SRCC=stats.spearmanr(P[k], y)[0], lo=np.percentile(B[k], 2.5), hi=np.percentile(B[k], 97.5)) for k in P]
    for k in ['PointNet-8192', 'Hybrid (PointNet + descriptors)']:
        dd = B[k] - B['CQI-learned']; rows.append(dict(method=f'{k} minus CQI-learned', SRCC=dd.mean(), lo=np.percentile(dd, 2.5), hi=np.percentile(dd, 97.5), p=2*min((dd<=0).mean(), (dd>=0).mean())))
    T = pd.DataFrame(rows); T.to_csv(D_PN + '/T_pointnet8192_hybrid.csv', index=False); display(T.round(3))
    pn = PN.train_predict(torch, Xt, lt, y, Xp[~th], ln[~th], epochs=40); hy = PN.train_predict(torch, Xt, Et, y, Xp[~th], E[~th], epochs=40)
    cq = d[~th][PN.PROPOSED].values @ PN.fit_nnls(dt[PN.PROPOSED].values, y); ys = d.RQ.values[~th]
    pd.DataFrame([dict(method=k, SRCC=stats.spearmanr(v, ys)[0]) for k, v in [('PointNet-8192', pn), ('Hybrid', hy), ('CQI-learned', cq)]]).to_csv(D_PN + '/T_pointnet8192_stanford.csv', index=False)

## Stage 6 · Exploratory unsupported surface fraction (USF)

In [ ]:
if RUN['usf']:
    rows = pd.read_csv(find('all_rows.csv')); rows = rows[rows.source != 'smoke']; dtu = pd.read_csv(find('dtu_all_rows.csv'))
    st = Rn.load_stanford(); ents = Rn.thingi_ids(N_THINGI, seed=SEED)
    Parallel(n_jobs=NCPU, verbose=5)([delayed(U.usf_object)(f'stanford_{k}', V, F, D_USF + '/usf_rows') for k, (V, F) in st.items()] +
                                     [delayed(lambda e: U.usf_object(f"thingi_{e['file_id']}", *Rn.load_thingi(e), D_USF + '/usf_rows'))(e) for e in ents])
    usf = pd.concat([pd.read_csv(f) for f in glob.glob(D_USF + '/usf_rows/*.csv')])
    Dm.download_all(Dm.SCANS)
    Parallel(n_jobs=NCPU, verbose=5)(delayed(U.usf_dtu_scan)(s, D_USF + '/usf_dtu') for s in Dm.SCANS)
    ud = pd.concat([pd.read_csv(f) for f in glob.glob(D_USF + '/usf_dtu/*.csv')]); DU = dtu.merge(ud, on=['scan', 'method'])
    print('DTU subsample sizes identical:', (DU.n_check == DU.n_roi).mean())
    RU = rows.merge(usf, on=['object', 'variant'])
    for k in U.KS:
        T, S = U.analyze(RU, DU, D_USF, k=k); print(f'--- k={k}'); display(T.round(3))

## Stage 7 · Figures (600 dpi PNG + TIFF)

In [ ]:
# Fig. 2: example defects on the Stanford Bunny
if RUN['figures']:
    import numpy as np, zlib, open3d as o3d, matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt, pandas as pd
    import prq_core as C
    from scipy.spatial import cKDTree
    W={"s_spacing":.2146,"s_density":0,"s_noise":.0362,"s_holes":.3336,"s_outliers":.0319,"s_normals":.0772,"s_msnd":.3067}
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":7,"savefig.dpi":600})
    m=o3d.io.read_triangle_mesh(o3d.data.BunnyMesh().path)
    V,F=C.normalize_mesh(np.asarray(m.vertices),np.asarray(m.triangles)); P0=C.sample_clean(V,F)
    G=C.sample_surface(V,F,C.CFG["N_EVAL"],seed=0); tG=cKDTree(G)
    rng=np.random.default_rng(zlib.crc32(b"stanford_Bunny|42")); vs={v[0]:v for v in C.make_variants(P0,rng,15)}
    pick=[("clean","Clean"),("noise_s0.75","Noise"),("sparsity_s0.75","Sparsity"),("nonuniform_s1.00","Non-uniform"),("holes_s1.00","Holes"),("outliers_s0.75","Outliers"),("ghosting_s1.00","Ghosting*")]
    f=plt.figure(figsize=(7.4,2.6)); rows=[]
    for j,(key,lab) in enumerate(pick):
        P=vs[key][3]; met=C.proposed_metrics(P); cqi=sum(W[k]*met[k] for k in W)
        Vr,Fr=C.reconstruct_poisson(P); ev=C.evaluate_mesh(Vr,Fr,G,tG,"",full_topology=False); rq=np.mean([ev[f"F@{t}"] for t in (0.5,1.0,2.0)])
        rows.append(dict(variant=key,n=len(P),CQI=cqi,RQ=rq))
        rot=lambda X: X[:,[0,2,1]]            # bunny y-up -> z-up
        ax=f.add_subplot(2,7,j+1,projection="3d"); Q=rot(P); 
        ax.scatter(Q[:,0],Q[:,1],Q[:,2],s=.05 if len(P)>5000 else .4,c=Q[:,2],cmap="viridis",linewidths=0,rasterized=True)
        ax.set_title(f"{lab}\nn={len(P):,}",fontsize=6.5,pad=0)
        mm=o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(Vr),o3d.utility.Vector3iVector(Fr))
        if len(Fr)>15000: mm=mm.simplify_quadric_decimation(15000)
        Vd=rot(np.asarray(mm.vertices)); Fd=np.asarray(mm.triangles)
        ax2=f.add_subplot(2,7,j+8,projection="3d")
        ax2.plot_trisurf(Vd[:,0],Vd[:,1],Vd[:,2],triangles=Fd,color="#c9d6e8",linewidth=0,shade=True,antialiased=False,rasterized=True)
        ax2.set_title(f"CQI {cqi:.2f}\nRQ {rq:.2f}",fontsize=6.2,pad=-2)
        for a in (ax,ax2):
            a.view_init(elev=12,azim=-70); a.set_axis_off(); a.set_box_aspect(None,zoom=1.38); lim=52; a.set_xlim(-lim,lim); a.set_ylim(-lim,lim); a.set_zlim(-lim,lim)
    f.subplots_adjust(left=.035,right=1,top=.88,bottom=0,wspace=-.05,hspace=.12)
    f.text(0.0,.6,"Raw cloud",rotation=90,fontsize=7,fontweight="bold"); f.text(0.0,.05,"Reconstruction",rotation=90,fontsize=7,fontweight="bold")
    f.savefig("/kaggle/working/figures/Fig2_examples.png",bbox_inches="tight"); f.savefig("/kaggle/working/figures/Fig2_examples.tif",bbox_inches="tight",pil_kwargs={"compression":"tiff_lzw"})
    print(pd.DataFrame(rows).round(3).to_string())


In [ ]:
# Figs. 1, 3-5, 8, 9 and S1
if RUN['figures']:
    import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from scipy import stats
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.spines.top":False,"axes.spines.right":False,
                         "savefig.dpi":600,"axes.titlesize":8.5,"axes.titleweight":"bold","legend.frameon":False})
    import glob, os
    def find(name):
        hits=sorted(glob.glob(f"/kaggle/working/**/{name}",recursive=True))+sorted(glob.glob(f"/kaggle/input/**/{name}",recursive=True))
        if not hits: raise FileNotFoundError(f"{name} not found - run the corresponding stage first")
        return hits[0]
    R=os.path.dirname(find("all_rows.csv"))+"/"; REV=os.path.dirname(find("T_closed_loop_policies.csv"))+"/"
    try: PNF=find("T_pointnet_cv.csv")
    except FileNotFoundError: PNF=None; print("Stage 5 (PointNet, GPU) not run: Fig. 5 is drawn without the PointNet bars")
    O="/kaggle/working/figures/"; os.makedirs(O,exist_ok=True)
    df=pd.read_csv(R+"all_rows.csv"); cv=pd.read_csv(R+"cv_predictions.csv"); B=pd.read_csv(R+"T_benchmark_cv.csv")
    D=pd.read_csv(find("dtu_all_rows.csv"))
    def save(f,n): f.savefig(O+n+".png",bbox_inches="tight"); f.savefig(O+n+".tif",bbox_inches="tight",pil_kwargs={"compression":"tiff_lzw"}); plt.close(f)
    def tag(ax,s): ax.text(-0.14,1.04,s,transform=ax.transAxes,fontweight="bold",fontsize=10)

    # ---- Fig 1 framework
    f,ax=plt.subplots(figsize=(7.2,2.9)); ax.axis("off"); ax.set_xlim(0,100); ax.set_ylim(-6,40)
    def box(x,y,w,h,t,c):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.4,rounding_size=1.2",fc=c,ec="#333",lw=.7))
        ax.text(x+w/2,y+h/2,t,ha="center",va="center",fontsize=6.2)
    def arr(x1,y1,x2,y2): ax.annotate("",(x2,y2),(x1,y1),arrowprops=dict(arrowstyle="-|>",lw=.8,color="#333"))
    box(1,26,17,11,"Reference mesh\n(Thingi10K / Stanford)","#e8eef7"); box(1,6,17,11,"Real MVS cloud\n(DTU, 3 methods)","#e8eef7")
    box(23,26,19,11,"Controlled corruption\n6 defect types × 5 levels\n+ mixed","#fdf1dc")
    box(48,26,21,11,"Raw point cloud\n(normalised to 100 mm)","#ffffff")
    box(48,6,21,13,"No-reference CQI\n7 descriptors (mm units)\n→ learned fusion","#dff0e0")
    box(75,26,24,11,"Frozen reconstruction\nScreened Poisson / BPA\n(no manual repair)","#f3e3f1")
    box(75,6,24,13,"Independent target RQ\nF-score vs reference\n@0.5/1/2 mm","#f3e3f1")
    arr(18,31.5,23,31.5); arr(42,31.5,48,31.5); arr(18,11.5,48,28); arr(69,31.5,75,31.5); arr(58.5,26,58.5,19.5)
    arr(87,26,87,19.5); arr(75,12.5,69,12.5)
    ax.text(72,1.0,"compare: SRCC / KRCC / PLCC,\nobject-disjoint CV",ha="center",fontsize=6,style="italic")
    ax.text(50,-4.5,"Evaluation never uses CQI to create labels: ground truth comes only from the reconstruction outcome.",ha="center",fontsize=6.5,style="italic")
    save(f,"Fig1_framework")

    # ---- Fig 2 noise calibration
    nz=df[df.type=="noise"]; g=nz.groupby("noise_sigma")
    f,ax=plt.subplots(figsize=(3.4,3.0))
    for col,lab,c,m in [("raw_noise_mm","Scale-extrapolated quadric (proposed)","C0","o"),("raw_noise_q24_mm","Single-scale quadric (k = 24)","C1","s")]:
        med=g[col].median(); q1=g[col].quantile(.25); q3=g[col].quantile(.75)
        ax.errorbar(med.index,med,yerr=[med-q1,q3-med],fmt=m+"-",c=c,ms=4,lw=1,capsize=2,label=lab)
    lim=[0.025,1.1]; ax.plot(lim,lim,"k--",lw=.7,label="Identity"); ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Injected noise σ (mm)"); ax.set_ylabel("Estimated σ (mm), median ± IQR"); ax.legend(fontsize=6.5,loc="upper left")
    save(f,"Fig3_noise_calibration")

    # ---- Fig 3 specificity effect size
    Z=pd.read_csv(R+"T_specificity_effectsize.csv",index_col=0)
    names={"raw_spacing_mm":"Spacing","raw_density":"Density uniformity","raw_noise_mm":"Noise (σ, mm)","raw_boundary_frac":"Hole boundary",
           "raw_outlier_frac":"Outliers","raw_normal_dev":"Normal consistency","raw_msnd":"MSND"}
    Z=Z.loc[list(names)]; V=np.sign(Z.values)*np.log10(1+np.abs(Z.values))
    f,ax=plt.subplots(figsize=(5.2,3.0)); im=ax.imshow(V,cmap="RdBu_r",vmin=-2,vmax=2,aspect="auto")
    ax.set_xticks(range(Z.shape[1])); ax.set_xticklabels([c.capitalize()+("*" if c=="ghosting" else "") for c in Z.columns],rotation=30,ha="right")
    ax.set_yticks(range(len(Z))); ax.set_yticklabels(names.values())
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]): ax.text(j,i,f"{Z.values[i,j]:.1f}",ha="center",va="center",fontsize=6.5,color="white" if abs(V[i,j])>1.2 else "black")
    cb=f.colorbar(im,ax=ax,shrink=.85); cb.set_label("sign·log10(1+|Δz|)",fontsize=7)
    ax.set_xlabel("Injected defect (maximum severity); *held-out type"); save(f,"Fig4_specificity")

    # ---- Fig 4 benchmark
    f=plt.figure(figsize=(7.2,5.6)); gs=f.add_gridspec(2,2,height_ratios=[1,1.05],hspace=.42,wspace=.35)
    ax=f.add_subplot(gs[0,:]); tag(ax,"(a)")
    if PNF is not None:
        PN=pd.read_csv(PNF); H=pd.read_csv(find("T_pointnet8192_hybrid.csv")); B=pd.concat([B,pd.DataFrame([dict(method="PointNet",SRCC=PN.SRCC[0],SRCC_lo=PN.lo[0],SRCC_hi=PN.hi[0]),dict(method="PointNet8192",SRCC=H.SRCC[0],SRCC_lo=H.lo[0],SRCC_hi=H.hi[0]),dict(method="Hybrid",SRCC=H.SRCC[1],SRCC_lo=H.lo[1],SRCC_hi=H.hi[1])])],ignore_index=True)
    sel=["GBT (black-box)","Hybrid","CQI-learned","CQI-adaptive","CQI-equal","single: s_msnd","single: s_holes","FR Chamfer (needs reference)","CQI-v1 (previous manuscript)","single: s_spacing","PointNet8192","PointNet","single: s_noise","Point count"]
    sel=[x for x in sel if x in set(B.method)]
    lab={"GBT (black-box)":"Gradient boosting (black box, upper bound)","CQI-learned":"CQI, learned weights (proposed)","CQI-adaptive":"CQI, geometry-conditioned weights",
         "CQI-equal":"CQI, equal weights","single: s_msnd":"Single descriptor: MSND","single: s_holes":"Single descriptor: hole boundary",
         "FR Chamfer (needs reference)":"Chamfer distance (full-reference)","CQI-v1 (previous manuscript)":"CQI-v1 (previous version)",
         "single: s_spacing":"Single descriptor: spacing","PointNet":"PointNet, 2048 points","PointNet8192":"PointNet, 8192 points","Hybrid":"PointNet + descriptors (hybrid)","single: s_noise":"Single descriptor: noise","Point count":"Point count"}
    b=B.set_index("method").loc[sel[::-1]]
    cols=["#2c7fb8" if "CQI-learned"==m else ("#999" if m in("GBT (black-box)","FR Chamfer (needs reference)") else ("#fdae6b" if m in ("PointNet","PointNet8192","Hybrid") else "#9ecae1")) for m in b.index]
    ax.barh([lab[m] for m in b.index],b.SRCC,xerr=[b.SRCC-b.SRCC_lo,b.SRCC_hi-b.SRCC],color=cols,ecolor="#333",capsize=2,height=.65)
    ax.set_xlabel("SRCC with reconstruction quality RQ (95% CI, object bootstrap)"); ax.set_xlim(0,.9)
    ax=f.add_subplot(gs[1,0]); tag(ax,"(b)")
    hb=ax.hexbin(cv["pred_CQI-learned"],cv.RQ,gridsize=45,cmap="Blues",bins="log",mincnt=1,linewidths=0)
    f.colorbar(hb,ax=ax,shrink=.85).set_label("clouds (log)",fontsize=7)
    ax.set_xlabel("CQI-learned (no reference)"); ax.set_ylabel("RQ (frozen Poisson pipeline)")
    ax=f.add_subplot(gs[1,1]); tag(ax,"(c)")
    W=pd.read_csv(R+"T_learned_weights.csv").drop(columns="fold")
    wl=["Spacing","Density","Noise","Holes","Outliers","Normals","MSND"]
    ax.bar(wl,W.mean(),yerr=W.std(),color="#41ab5d",capsize=2); ax.set_ylabel("Learned weight (mean ± SD, 5 folds)"); plt.setp(ax.get_xticklabels(),rotation=35,ha="right")
    save(f,"Fig5_benchmark")

    # ---- Fig 5 attribution
    sev,cm,_=learned_attribution(df)
    f,axs=plt.subplots(1,2,figsize=(7.0,2.9),gridspec_kw=dict(width_ratios=[1,1.15],wspace=.45))
    ax=axs[0]; tag(ax,"(a)")
    ax.plot(sev.index,sev.rule,"s--",c="C1",label="Largest z-score (rule)"); ax.plot(sev.index,sev.learned,"o-",c="C0",label="Logistic model (learned)")
    ax.axhline(.2,c="k",ls=":",lw=.7); ax.text(.02,.215,"chance (5 classes)",fontsize=6); ax.set_ylim(0,1)
    ax.set_xlabel("Injected severity level"); ax.set_ylabel("Top-1 attribution accuracy"); ax.legend(fontsize=6.5,loc="lower right")
    ax=axs[1]; tag(ax,"(b)"); labs=["Noise","Sparsity","Non-uniform","Holes","Outliers"]
    ax.imshow(cm,cmap="Blues",vmin=0,vmax=1); ax.set_xticks(range(5)); ax.set_xticklabels(labs,rotation=35,ha="right"); ax.set_yticks(range(5)); ax.set_yticklabels(labs)
    for i in range(5):
        for j in range(5): ax.text(j,i,f"{cm[i,j]:.2f}",ha="center",va="center",fontsize=6.5,color="white" if cm[i,j]>.6 else "black")
    ax.set_xlabel("Attributed defect (learned)"); ax.set_ylabel("Injected defect (severity ≥ 0.5)")
    save(f,"Fig8_attribution")

    # ---- Fig 6 closed loop
    PA=pd.read_csv(R+"T_closed_loop_per_action.csv",index_col=0)
    order=["outliers","denoise","uniformize","outliers+denoise","outliers+uniformize","denoise+uniformize","outliers+denoise+uniformize"]
    PA=PA[order].loc[["clean","noise","nonuniform","outliers","mixed"]]
    S=pd.read_csv(R+"T_closed_loop_summary.csv"); S2=pd.read_csv(R+"T_closed_loop_summary_RQ_BPA.csv")
    f,axs=plt.subplots(1,2,figsize=(7.4,3.0),gridspec_kw=dict(width_ratios=[1.5,1],wspace=1.05))
    ax=axs[0]; tag(ax,"(a)"); im=ax.imshow(PA.values,cmap="RdBu",vmin=-.08,vmax=.08,aspect="auto")
    ax.set_xticks(range(len(order))); ax.set_xticklabels(["SOR","Denoise","Uniformise","SOR+Den","SOR+Uni","Den+Uni","All three"],rotation=35,ha="right")
    ax.set_yticks(range(5)); ax.set_yticklabels(["Clean","Noise","Non-uniform","Outliers","Mixed"])
    for i in range(PA.shape[0]):
        for j in range(PA.shape[1]): ax.text(j,i,f"{PA.values[i,j]:+.2f}",ha="center",va="center",fontsize=6,color="white" if abs(PA.values[i,j])>.045 else "black")
    cb=f.colorbar(im,ax=ax,shrink=.85); cb.set_label("Mean ΔRQ (Poisson)",fontsize=7)
    ax=axs[1]; tag(ax,"(b)")
    SR=pd.read_csv(REV+"T_closed_loop_policies.csv"); SR=SR[SR.target=="RQ"].set_index("strategy")
    st=[("all","Apply all"),("random_single","Random single"),("z>3 (fold-wise)","z-score rule"),("learned detector","Learned detector"),("learned detector, SOR only","Learned, SOR only*"),("oracle","Oracle")]
    y=np.arange(len(st))[::-1]
    ax.barh(y+.2,[SR.loc[k,"harmed"] for k,_ in st],.4,color="#d6604d",label="harmed")
    ax.barh(y-.2,[SR.loc[k,"improved"] for k,_ in st],.4,color="#4393c3",label="improved")
    ax.set_yticks(y); ax.set_yticklabels([l for _,l in st]); ax.set_xlabel("Fraction of clouds"); ax.legend(fontsize=6.5,loc="lower right")
    save(f,"FigS1_closed_loop")

    # ---- Fig 9 DTU (22 standard scenes)
    def draw(D, out):
      with plt.rc_context({"font.family":"DejaVu Sans","font.size":7.5,"savefig.dpi":600,"axes.spines.top":False,"axes.spines.right":False,"axes.linewidth":0.8,"xtick.labelsize":7.5,"ytick.labelsize":7.5,"axes.labelsize":7.5}):
          D=D.copy(); D["RQ_BPA"]=D[["BPA_F@0.5","BPA_F@1.0","BPA_F@2.0"]].mean(axis=1); D["RQ"]=D[["PSR_F@0.5","PSR_F@1.0","PSR_F@2.0"]].mean(axis=1)
          D["negacc"]=-D.DTU_acc; D["negcomp"]=-D.DTU_comp
          def pw(a,b):
              ok=tot=0
              for _,g in D.groupby("scan"):
                  x=g[a].values;y=g[b].values
                  for i in range(len(x)):
                      for j in range(i+1,len(x)):
                          if y[i]!=y[j]: tot+=1; ok+=(x[i]-x[j])*(y[i]-y[j])>0
              return ok/tot
          f=plt.figure(figsize=(7.2,3.3))
          ax=f.add_axes([0.075,0.15,0.37,0.68])                       # panel (a)
          for m,mk,lb,c in [("camp","o","Campbell et al.","#1f77b4"),("furu","s","Furukawa & Ponce","#ff7f0e"),("tola","^","Tola et al.","#2ca02c")]:
              g=D[D.method==m]; ax.scatter(g.CQI_learned,g.RQ_BPA,s=18,marker=mk,label=lb,alpha=.85,color=c,linewidths=0)
          ax.set_xlabel("CQI-learned (frozen, trained on Thingi10K)"); ax.set_ylabel("RQ, ball pivoting (DTU masks)")
          ax.legend(fontsize=7,ncol=3,loc="lower left",bbox_to_anchor=(-0.02,1.02),frameon=False,handletextpad=0.2,columnspacing=1.0)
          f.text(0.005,0.95,"(a)",fontsize=9,fontweight="bold")
          ax2=f.add_axes([0.725,0.15,0.255,0.68])                     # panel (b), wide gap reserved for its labels
          items=[("CQI vs −accuracy","CQI_learned","negacc"),("CQI vs −completeness","CQI_learned","negcomp"),("CQI vs RQ (ball pivoting)","CQI_learned","RQ_BPA"),
                 ("CQI vs RQ (Poisson)","CQI_learned","RQ"),("RQ (Poisson) vs RQ (ball pivoting)","RQ","RQ_BPA")]
          v=[pw(a,b) for _,a,b in items]; lab=[i[0] for i in items]
          col=["#2c7fb8"]*4+["#9a9a9a"]; y=np.arange(len(items))[::-1]
          ax2.barh(y,v,color=col,height=0.62)
          ax2.set_yticks(y); ax2.set_yticklabels(lab,fontsize=7)
          ax2.axvline(.5,c="k",ls=":",lw=.8); ax2.set_ylim(-0.6,len(items)-0.1)
          ax2.text(.5,len(items)-0.15,"chance",fontsize=6.5,ha="center",va="bottom")
          for yy,val in zip(y,v): ax2.text(val+.015,yy,f"{val:.2f}",va="center",fontsize=7)
          ax2.set_xlim(0,1.05); ax2.set_xlabel("Within-scene pairwise\nranking agreement")
          f.text(0.455,0.95,"(b)",fontsize=9,fontweight="bold")
          f.savefig(out+".png",bbox_inches="tight",pad_inches=0.03); f.savefig(out+".tif",bbox_inches="tight",pad_inches=0.03,pil_kwargs={"compression":"tiff_lzw"})
          return dict(zip(lab,[round(x,2) for x in v]))
    print("DTU pairwise:", draw(D, O+"Fig9_dtu"))


In [ ]:
# Fig. 6: ranking examples on held-out Stanford objects
if RUN['figures']:
    import numpy as np, zlib, open3d as o3d, matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt, prq_core as C
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    def shaded_mesh(ax,V,F,el,az):
        T=V[F]; n=np.cross(T[:,1]-T[:,0],T[:,2]-T[:,0]); n/=np.linalg.norm(n,axis=1,keepdims=True)+1e-12
        e,a=np.radians(el),np.radians(az); L=np.array([np.cos(e)*np.cos(a),np.cos(e)*np.sin(a),np.sin(e)])
        L2=np.array([-L[1],L[0],0.6]); L2/=np.linalg.norm(L2)
        k=0.30+0.55*np.abs(n@L)+0.15*np.abs(n@L2)          # orientation-independent shading
        base=np.array([0.80,0.85,0.92]); col=np.clip(base[None]*k[:,None],0,1)
        pc=Poly3DCollection(T,facecolors=col,edgecolors="none",linewidths=0,rasterized=True); ax.add_collection3d(pc)
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":7,"savefig.dpi":600})
    SEL={"Bunny":(o3d.data.BunnyMesh,[("mixed_14",0.806,0.947,"outliers + noise"),("mixed_04",0.419,0.836,"sparsity + strong noise"),("mixed_01",0.153,0.595,"holes + severe sparsity")]),
         "Armadillo":(o3d.data.ArmadilloMesh,[("mixed_05",0.815,0.970,"mild holes + noise"),("mixed_04",0.497,0.752,"holes + non-uniform + outliers"),("mixed_14",0.083,0.482,"sparsity + non-uniform + outliers + noise")])}
    VIEW={"Bunny":(12,-70,lambda X:X[:,[0,2,1]]),"Armadillo":(10,-90,lambda X:X[:,[0,2,1]]*np.array([1,-1,1]))}
    f=plt.figure(figsize=(7.2,6.4)); r0=0
    for name,(ds,picks) in SEL.items():
        m=o3d.io.read_triangle_mesh(ds().path); V,F=C.normalize_mesh(np.asarray(m.vertices),np.asarray(m.triangles)); P0=C.sample_clean(V,F)
        rng=np.random.default_rng(zlib.crc32(f"stanford_{name}|42".encode())); vs={v[0]:v[3] for v in C.make_variants(P0,rng,15)}
        el,az,rot=VIEW[name]
        for j,(var,cqi,rq,lab) in enumerate(picks):
            P=vs[var]; Vr,Fr=C.reconstruct_poisson(P)
            mm=o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(Vr),o3d.utility.Vector3iVector(Fr))
            if len(Fr)>15000: mm=mm.simplify_quadric_decimation(15000)
            Vd=rot(np.asarray(mm.vertices)); Fd=np.asarray(mm.triangles); Q=rot(P)
            ax=f.add_subplot(4,3,r0*3+j+1,projection="3d")
            ax.scatter(Q[:,0],Q[:,1],Q[:,2],s=.06 if len(P)>5000 else .5,c=Q[:,2],cmap="viridis",linewidths=0,rasterized=True)
            ax.set_title(f"{['High','Medium','Low'][j]} CQI ({cqi:.2f}) | n = {len(P):,}\n{lab}",fontsize=6.3,pad=0)
            ax2=f.add_subplot(4,3,(r0+1)*3+j+1,projection="3d")
            shaded_mesh(ax2,Vd,Fd,el,az)
            ax2.set_title(f"Reconstruction RQ = {rq:.2f}",fontsize=6.3,pad=0)
            for a in (ax,ax2):
                a.view_init(elev=el,azim=az); a.set_axis_off(); a.set_box_aspect(None,zoom=1.35); lim=52
                a.set_xlim(-lim,lim); a.set_ylim(-lim,lim); a.set_zlim(-lim,lim)
        r0+=2
    f.subplots_adjust(left=.04,right=1,top=.95,bottom=0,wspace=-.05,hspace=.18)
    for y,t in [(.86,"Bunny: cloud"),(.63,"Reconstruction"),(.38,"Armadillo: cloud"),(.14,"Reconstruction")]:
        f.text(0,y,t,rotation=90,fontsize=7,fontweight="bold",va="center")
    f.savefig("/kaggle/working/figures/Fig6_ranking_examples.png",bbox_inches="tight"); f.savefig("/kaggle/working/figures/Fig6_ranking_examples.tif",bbox_inches="tight",pil_kwargs={"compression":"tiff_lzw"})


In [ ]:
# Fig. 7: per-point descriptor maps
if RUN['figures']:
    import numpy as np, pandas as pd, zlib, open3d as o3d, matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt, prq_core as C, figlib
    from scipy.spatial import cKDTree
    from sklearn.linear_model import LogisticRegression
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":7,"savefig.dpi":600})
    # exact study cloud
    m=o3d.io.read_triangle_mesh(o3d.data.BunnyMesh().path); V,F=C.normalize_mesh(np.asarray(m.vertices),np.asarray(m.triangles)); P0=C.sample_clean(V,F)
    rng=np.random.default_rng(zlib.crc32(b"stanford_Bunny|42")); P={v[0]:v[3] for v in C.make_variants(P0,rng,15)}["mixed_10"]
    tree=cKDTree(P)
    # per-point maps
    def quad_res(k):
        out=np.empty(len(P))
        for s,e in C._chunks(len(P)):
            _,ix=tree.query(P[s:e],k=k,workers=-1); u,w,h=C._local_uvh(P[ix])
            A=np.stack([u**2,u*w,w**2,u,w,np.ones_like(u)],-1)
            c=np.linalg.solve(np.einsum("mki,mkj->mij",A,A)+1e-9*np.eye(6),np.einsum("mki,mk->mi",A,h)[...,None])[...,0]
            r=h-np.einsum("mki,mi->mk",A,c); out[s:e]=np.sqrt((r**2).sum(1)/(k-6))
        return out
    noise=quad_res(12)
    B=C._boundary_points(P,tree,16,2*np.pi/3); _,ixa=tree.query(P,k=9,workers=-1); hole=B&(B[ixa[:,1:]].sum(1)>=2)
    d8=tree.query(P,k=9,workers=-1)[0][:,1:].mean(1); outl=d8>3*np.median(d8)
    N10=C.all_normals(P,tree,10); N40=C.all_normals(P,tree,40); msnd=1-np.abs((N10*N40).sum(1))
    # learned detectors (trained on all synthetic clouds of the benchmark) applied to this cloud's descriptors
    df=pd.read_csv(find("all_rows.csv")); ref=figlib.clean_ref(df); labs=["noise","sparsity","nonuniform","holes","outliers"]
    def Y(X): return np.column_stack([((X.type==t)|((X.type=="mixed")&(X.get(f"sev_{t}",pd.Series(np.nan,index=X.index))>=.25))).values for t in labs])
    tr=df.type.isin(labs+["mixed","clean"]); row=df[(df.object=="stanford_Bunny")&(df.variant=="mixed_10")]
    tr=tr&(df.object!="stanford_Bunny")            # exclude this object from training
    prob=[LogisticRegression(max_iter=3000).fit(figlib._zf(df[tr],ref),Y(df[tr])[:,j]).predict_proba(figlib._zf(row,ref))[0,1] for j in range(5)]
    print("detector probabilities:",dict(zip(labs,np.round(prob,3))), "| n",len(P),"| hole pts",hole.sum(),"| outliers",outl.sum())
    Vr,Fr=C.reconstruct_poisson(P)
    rot=lambda X:X[:,[0,2,1]]; el,az=12,-70; Q=rot(P)
    f=plt.figure(figsize=(7.2,4.6))
    def cloud_ax(i,title):
        ax=f.add_subplot(2,3,i,projection="3d"); ax.set_title(title,fontsize=7,pad=0); return ax
    def fin(ax):
        ax.view_init(elev=el,azim=az); ax.set_axis_off(); ax.set_box_aspect(None,zoom=1.35); lim=58
        ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim); ax.set_zlim(-lim,lim)
    ax=cloud_ax(1,"(a) Raw cloud (holes + outliers)"); ax.scatter(Q[:,0],Q[:,1],Q[:,2],s=.06,c=Q[:,2],cmap="viridis",linewidths=0,rasterized=True); fin(ax)
    ax=cloud_ax(2,"(b) Local noise (quadric residual, mm)")
    sc=ax.scatter(Q[:,0],Q[:,1],Q[:,2],s=.08,c=np.clip(noise,0,0.5),cmap="inferno_r",vmin=0,vmax=0.5,linewidths=0,rasterized=True); fin(ax)
    cb=f.colorbar(sc,ax=ax,shrink=.45,pad=0); cb.ax.tick_params(labelsize=5.5)
    ax=cloud_ax(3,"(c) Hole rims (red) and outliers (black)")
    ok=~(hole|outl); ax.scatter(Q[ok,0],Q[ok,1],Q[ok,2],s=.12,c="#a8a8a8",linewidths=0,rasterized=True)
    ax.scatter(Q[hole,0],Q[hole,1],Q[hole,2],s=1.2,c="#d62728",linewidths=0,rasterized=True)
    ax.scatter(Q[outl,0],Q[outl,1],Q[outl,2],s=1.2,c="k",linewidths=0,rasterized=True); fin(ax)
    ax=cloud_ax(4,"(d) Multi-scale normal deviation")
    sc=ax.scatter(Q[:,0],Q[:,1],Q[:,2],s=.08,c=np.clip(msnd,0,0.3),cmap="magma_r",vmin=0,vmax=0.3,linewidths=0,rasterized=True); fin(ax)
    cb=f.colorbar(sc,ax=ax,shrink=.45,pad=0); cb.ax.tick_params(labelsize=5.5)
    ax=cloud_ax(5,"(e) Frozen Poisson reconstruction")
    mm=o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(Vr),o3d.utility.Vector3iVector(Fr)).simplify_quadric_decimation(15000)
    T=rot(np.asarray(mm.vertices))[np.asarray(mm.triangles)]; n=np.cross(T[:,1]-T[:,0],T[:,2]-T[:,0]); n/=np.linalg.norm(n,axis=1,keepdims=True)+1e-12
    e,a=np.radians(el),np.radians(az); L=np.array([np.cos(e)*np.cos(a),np.cos(e)*np.sin(a),np.sin(e)])
    k=0.35+0.65*np.abs(n@L); ax.add_collection3d(Poly3DCollection(T,facecolors=np.clip(np.array([.80,.85,.92])[None]*k[:,None],0,1),edgecolors="none",rasterized=True)); fin(ax)
    ax=f.add_subplot(2,3,6); ax.barh(["Noise","Sparsity","Non-uniform","Holes","Outliers"][::-1],prob[::-1],color=["#d62728" if p>0.5 else "#9ecae1" for p in prob[::-1]])
    ax.axvline(.5,c="k",ls=":",lw=.7); ax.set_xlim(0,1); ax.set_xlabel("Detection probability",fontsize=6.5); ax.set_title("(f) Learned defect detection",fontsize=7)
    ax.tick_params(labelsize=6); ax.spines[["top","right"]].set_visible(False)
    f.subplots_adjust(left=.01,right=.97,top=.93,bottom=.06,wspace=.08,hspace=.18)
    f.savefig("/kaggle/working/figures/Fig7_descriptor_maps.png",bbox_inches="tight"); f.savefig("/kaggle/working/figures/Fig7_descriptor_maps.tif",bbox_inches="tight",pil_kwargs={"compression":"tiff_lzw"})


In [ ]:
# Fig. 10: frontal view of DTU scan 33
if RUN['figures']:
    Dm.fetch(Dm.URL['sample'], {33}); Dm.fetch(Dm.URL['points'], {33}); Dm.fetch(Dm.URL['mvs'], {33})
    Q.dtu_figure(33, D_FIG, frontal=True)
    for ext in ('png', 'tif'): os.replace(f'{D_FIG}/Fig_dtu_examples_scan33_frontal1.{ext}', f'{D_FIG}/Fig10_dtu_examples.{ext}')
    print(sorted(os.listdir(D_FIG)))

## Package outputs

In [ ]:
from IPython.display import FileLink, display
!cd /kaggle/working && zip -qr pointcloud_quality_outputs.zip rows results dtu_rows dtu_results dtu_extended extras pointnet usf figures thingi_selection.csv 2>/dev/null; ls -lh pointcloud_quality_outputs.zip
os.chdir(W); display(FileLink('pointcloud_quality_outputs.zip'))